In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [1/10] 環境セットアップ
# ------------------------------------------------------------
# ランタイム起動後に1回だけ実行する。
#
# Playwright のブラウザ本体（chromium）は、JavaScriptで組み立てられる
# ページを読むときだけ必要になる。数十秒かかるので既定では入れない。
# 静的HTMLで取れなかったときに INSTALL_PLAYWRIGHT_BROWSER を True にして
# このセルだけ流し直す。
# ============================================================
!pip install -q requests beautifulsoup4 lxml ipywidgets google-api-python-client google-auth-httplib2 google-auth-oauthlib playwright nest_asyncio

INSTALL_PLAYWRIGHT_BROWSER = False  # @param {type:"boolean"}
if INSTALL_PLAYWRIGHT_BROWSER:
    !python -m playwright install --with-deps chromium > /dev/null

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [2/10] 設定① 対戦カード
# ------------------------------------------------------------
# 最初に決めるのはこの2つだけ。
# 「長崎」「グランパス」のような短縮名・愛称でも、半角全角どちらでも通る。
#
# ここを入れたら [3/10] → [4/10] と実行する。[4/10] がJリーグ公式の
# 日程から該当カードを探し、キックオフ・会場・中継の初期値を入れる。
# ============================================================

MY_TEAM = "名古屋グランパス"  # @param {type:"string"}
OPPONENT_TEAM = "FC東京"  # @param {type:"string"}

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [3/10] 共通基盤（import / ログ / 実行レポート / 通信）
# ------------------------------------------------------------
# Konwenga Helper Ver.9.0.0 の [2/8] をそのまま転用している。
# 直す場合は両方そろえること。
# ============================================================
from __future__ import annotations

import asyncio
import logging
import re
import sys
import threading
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import date, datetime
from enum import Enum
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Final, List, Optional, Sequence, Tuple, Union
from urllib import robotparser
from urllib.parse import parse_qs, urljoin, urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup, NavigableString

try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

try:
    from google.colab import auth
    from google.auth import default
    from googleapiclient.discovery import build
    _IN_COLAB = True
except Exception:
    _IN_COLAB = False

VERSION: Final[str] = "1.6.1"

logger = logging.getLogger("dankoba")

_QUIET_LOGGERS: Final[Tuple[str, ...]] = (
    "googleapiclient.discovery_cache",
    "google_auth_httplib2",
    "urllib3",
)


class _NotebookLogHandler(logging.StreamHandler):
    """Colab はセルごとに sys.stdout を差し替えるため、書き出し時に解決する。"""

    @property
    def stream(self):
        return sys.stdout

    @stream.setter
    def stream(self, value):
        # StreamHandler.__init__ からの代入は無視する
        pass


def setup_logging(level: str = "INFO") -> None:
    """root のハンドラは触らない。Colab 側のハンドラを外しに行くと
    出力待ちのロックを掴んでセルが固まることがある。"""
    resolved_level = getattr(logging, str(level).strip().upper(), logging.INFO)
    if not any(isinstance(handler, _NotebookLogHandler) for handler in logger.handlers):
        handler = _NotebookLogHandler()
        handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
        logger.addHandler(handler)
    logger.setLevel(resolved_level)
    logger.propagate = False
    for name in _QUIET_LOGGERS:
        logging.getLogger(name).setLevel(logging.WARNING)


setup_logging("INFO")


def normalize_club_key(value: str) -> str:
    """NFKC正規化で全角と半角の差を吸収し、大文字小文字も無視する。
    「ＦＣ東京」と「FC東京」、「Ｖ・ファーレン長崎」と「V・ファーレン長崎」が
    同じキーになる。クラブ情報 [5/10] より先の [4/10] でも使うのでここに置く。"""
    return unicodedata.normalize("NFKC", str(value or "")).strip().casefold()


# 旧名（アップロード版モジュールからの移植互換）
_normalise = normalize_club_key


def display_width(text: str) -> int:
    """全角を2、半角を1として数える。ログの列を目視で揃えるため。"""
    return sum(2 if unicodedata.east_asian_width(char) in ("W", "F") else 1 for char in str(text))


def pad_display(text: str, width: int, align: str = "left") -> str:
    padding = " " * max(0, width - display_width(text))
    return padding + str(text) if align == "right" else str(text) + padding


# ------------------------------------------------------------
# 実行結果サマリ
# ------------------------------------------------------------
class StepStatus(str, Enum):
    OK = "OK"
    NG = "NG"
    SKIP = "SKIP"


@dataclass
class StepResult:
    name: str
    status: StepStatus
    detail: str = ""


class RunReport:
    def __init__(self) -> None:
        self.steps: List[StepResult] = []
        self.warnings: List[str] = []
        self.document_url: str = ""

    def record(self, name: str, status: StepStatus, detail: str = "") -> None:
        self.steps.append(StepResult(name=name, status=status, detail=detail))

    def ok(self, name: str, detail: str = "") -> None:
        self.record(name, StepStatus.OK, detail)

    def ng(self, name: str, detail: str = "") -> None:
        self.record(name, StepStatus.NG, detail)

    def skip(self, name: str, detail: str = "") -> None:
        self.record(name, StepStatus.SKIP, detail)

    def warn(self, message: str) -> None:
        logger.warning(message)
        self.warnings.append(message)

    @property
    def failed_steps(self) -> List[StepResult]:
        return [step for step in self.steps if step.status is StepStatus.NG]

    def render(self) -> str:
        name_width = max([display_width(step.name) for step in self.steps] + [16])
        lines = ["", "=== 実行結果サマリ ==="]
        for step in self.steps:
            detail = f"  {step.detail}" if step.detail else ""
            lines.append(f"{pad_display(step.name, name_width)}  {step.status.value:<4}{detail}")
        if self.document_url:
            lines.append(f"{pad_display('Docs URL', name_width)}  {'':<4}  {self.document_url}")
        if self.warnings:
            lines.append("")
            lines.append(f"警告 {len(self.warnings)}件:")
            lines.extend(f"  - {message}" for message in self.warnings)
        else:
            lines.append("")
            lines.append("警告なし")
        failed = self.failed_steps
        lines.append("")
        if failed:
            lines.append(f"失敗 {len(failed)}件: {', '.join(step.name for step in failed)}")
        elif self.warnings:
            lines.append("失敗はありませんが、警告の内容を確認してください。")
        else:
            lines.append("テンプレートを生成しました。")
        lines.append("=" * 22)
        return "\n".join(lines)

    def emit(self) -> None:
        logger.info(self.render())


def emit_warning(report: Optional[RunReport], message: str, *args: Any) -> None:
    text = message % args if args else message
    if report is not None:
        report.warn(text)
    else:
        logger.warning(text)


# ------------------------------------------------------------
# 通信基盤
# ------------------------------------------------------------
@dataclass(frozen=True)
class NetworkConfig:
    timeout_sec: int = 20
    playwright_timeout_sec: int = 30
    max_workers: int = 4
    user_agent: str = "Dankoba-Helper/1.6 (+https://grapo.net/)"
    default_encoding: str = "utf-8"
    # 同じホストへ連続で投げるときの最短間隔。相手は個人運営に近いクラブもある。
    min_interval_sec: float = 1.0
    respect_robots: bool = True


def build_session(network: NetworkConfig) -> requests.Session:
    """一時エラーだけ再試行するセッション。Konwenga [3/9] からの移植。"""
    session = requests.Session()
    retry = Retry(
        total=3, connect=3, read=3, status=3,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "HEAD"}),
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=max(4, network.max_workers),
        pool_maxsize=max(8, network.max_workers * 2),
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({"User-Agent": network.user_agent})
    return session


def decode_response_text(response: requests.Response, network: NetworkConfig) -> str:
    """apparent_encoding はレスポンス全文を chardet で走査するので使わない。
    ただしクラブ公式サイトは Football LAB と違って文字コードがまちまちなので、
    サーバが宣言していればそれを優先し、無い場合だけ既定値に落とす。"""
    declared = (response.encoding or "").strip().lower()
    if not declared or declared == "iso-8859-1":
        # requests は Content-Type に charset が無いと ISO-8859-1 を入れてくる
        meta_encoding = requests.utils.get_encodings_from_content(response.text[:2048])
        response.encoding = meta_encoding[0] if meta_encoding else network.default_encoding
    return response.text


def run_in_notebook(coro):
    """nest_asyncio.apply() 済みなので asyncio.run で足りる。
    get_event_loop() は Python 3.12 以降で非推奨。"""
    return asyncio.run(coro)


class HostThrottle:
    """ホストごとに最短間隔を空ける。並列取得でも1ホストに集中させない。"""

    def __init__(self, min_interval_sec: float):
        self.min_interval_sec = max(0.0, float(min_interval_sec))
        self._last_access: Dict[str, float] = {}
        self._lock = threading.Lock()

    def wait(self, url: str) -> None:
        if self.min_interval_sec <= 0:
            return
        host = urlparse(url).netloc.lower()
        with self._lock:
            previous = self._last_access.get(host, 0.0)
            elapsed = time.monotonic() - previous
            sleep_sec = self.min_interval_sec - elapsed
            if sleep_sec > 0:
                time.sleep(sleep_sec)
            self._last_access[host] = time.monotonic()


class RobotsPolicy:
    """robots.txt をホストごとに1回だけ読んで判定を覚える。

    取得できなかった場合は許可として扱う。robots.txt が無いサイトを
    一律で拒否すると、実質すべて読めなくなるため。"""

    def __init__(
        self,
        session: requests.Session,
        network: NetworkConfig,
        report: Optional[RunReport] = None,
    ):
        self.session = session
        self.network = network
        self.report = report
        self._parsers: Dict[str, Optional[robotparser.RobotFileParser]] = {}
        self._lock = threading.Lock()

    def _parser_for(self, url: str) -> Optional[robotparser.RobotFileParser]:
        parsed = urlparse(url)
        origin = f"{parsed.scheme}://{parsed.netloc}"
        with self._lock:
            if origin in self._parsers:
                return self._parsers[origin]
        parser: Optional[robotparser.RobotFileParser] = None
        try:
            response = self.session.get(urljoin(origin, "/robots.txt"), timeout=self.network.timeout_sec)
            if response.status_code == 200:
                parser = robotparser.RobotFileParser()
                parser.parse(decode_response_text(response, self.network).splitlines())
            else:
                logger.debug("robots.txt が見つかりません (HTTP %s): %s", response.status_code, origin)
        except Exception:
            logger.debug("robots.txt の取得に失敗しました: %s", origin, exc_info=True)
        with self._lock:
            self._parsers[origin] = parser
        return parser

    def is_allowed(self, url: str) -> bool:
        if not self.network.respect_robots:
            return True
        parser = self._parser_for(url)
        if parser is None:
            return True
        return parser.can_fetch(self.network.user_agent, url)


@dataclass
class FetchResult:
    url: str
    html: str
    source: str  # "静的HTML" または "Playwright"
    status_code: Optional[int] = None
    soup: Optional[BeautifulSoup] = None

    def links(self, keywords: Sequence[str], limit: int = 5) -> List[Tuple[str, str]]:
        return extract_links(self.soup, self.url, keywords, limit=limit)


def parse_html(html: str) -> BeautifulSoup:
    """lxml が使えればそちらを使う。入れ子の壊れたHTMLの補正が素直なため。"""
    try:
        return BeautifulSoup(html, "lxml")
    except Exception:
        logger.debug("lxmlが使えないためhtml.parserで解析します", exc_info=True)
        return BeautifulSoup(html, "html.parser")


def extract_links(
    soup: Optional[BeautifulSoup],
    base_url: str,
    keywords: Sequence[str],
    limit: int = 5,
) -> List[Tuple[str, str]]:
    """アンカーの文字列かURLにキーワードを含むリンクを拾い、絶対URLにして返す。
    同じURLは1回だけ。順序はページ上の並びを保つ。"""
    if soup is None:
        return []
    lowered = [str(keyword).lower() for keyword in keywords if str(keyword).strip()]
    found: List[Tuple[str, str]] = []
    seen: set = set()
    for anchor in soup.find_all("a", href=True):
        href = str(anchor.get("href")).strip()
        if not href or href.startswith(("#", "javascript:", "mailto:", "tel:")):
            continue
        text = anchor.get_text(" ", strip=True)
        haystack = f"{text} {href}".lower()
        if not any(keyword in haystack for keyword in lowered):
            continue
        absolute = urljoin(base_url, href)
        if absolute in seen:
            continue
        seen.add(absolute)
        found.append((text or absolute, absolute))
        if len(found) >= limit:
            break
    return found


class PageFetcher:
    """静的HTMLで取り、だめなら Playwright に落とす。

    Konwenga の FootballLabShotsSvgExporter が取っていた形をそのまま一般化した。
    シュートチャートのSVGがJavaScriptで描かれていたのと同じ事情が、
    クラブ公式サイトの試合情報ページでも起きる。"""

    def __init__(
        self,
        session: requests.Session,
        network: Optional[NetworkConfig] = None,
        report: Optional[RunReport] = None,
    ):
        self.session = session
        self.network = network or NetworkConfig()
        self.report = report
        self.throttle = HostThrottle(self.network.min_interval_sec)
        self.robots = RobotsPolicy(session, self.network, report)

    def _warn(self, message: str, *args: Any) -> None:
        emit_warning(self.report, message, *args)

    def fetch(
        self,
        url: str,
        *,
        use_playwright_fallback: bool = False,
        wait_selector: str = "",
        min_html_length: int = 0,
        force_render: bool = False,
    ) -> Optional[FetchResult]:
        """1ページ取る。取れなければ None を返して警告する。

        min_html_length を指定すると、本文が短すぎる（=JavaScriptで組み立てる
        ページを空のまま受け取った）場合に Playwright へ回す。
        force_render を立てると、静的HTMLを試さず最初からブラウザで描画する。
        静的HTMLは取れているのに一部の要素だけ入っていない、というときに使う。"""
        if not str(url or "").strip():
            return None
        if not self.robots.is_allowed(url):
            self._warn("robots.txt で許可されていないため取得しません: %s", url)
            return None
        if force_render:
            self.throttle.wait(url)
            return self._fetch_rendered(url, wait_selector)

        result: Optional[FetchResult] = None
        self.throttle.wait(url)
        try:
            response = self.session.get(url, timeout=self.network.timeout_sec)
            response.raise_for_status()
            html = decode_response_text(response, self.network)
            result = FetchResult(url=url, html=html, source="静的HTML",
                                 status_code=response.status_code, soup=parse_html(html))
        except requests.HTTPError as error:
            status = getattr(getattr(error, "response", None), "status_code", "不明")
            self._warn("ページを取得できませんでした (HTTP %s): %s", status, url)
        except requests.RequestException:
            logger.exception("ページ取得に失敗しました（接続またはタイムアウト）: %s", url)
            self._warn("ページを取得できませんでした（接続またはタイムアウト）: %s", url)
        except Exception:
            logger.exception("ページ解析に失敗しました: %s", url)
            self._warn("ページを解析できませんでした: %s", url)

        needs_fallback = use_playwright_fallback and (
            result is None or (min_html_length and len(result.html) < min_html_length)
        )
        if not needs_fallback:
            return result

        logger.info("静的HTMLでは内容が取れないため Playwright に切り替えます: %s", url)
        rendered = self._fetch_rendered(url, wait_selector)
        return rendered or result

    def _fetch_rendered(self, url: str, wait_selector: str = "") -> Optional[FetchResult]:
        try:
            html = run_in_notebook(self._render_async(url, wait_selector))
        except ModuleNotFoundError:
            self._warn("Playwright が入っていません。セル[1/10]の INSTALL_PLAYWRIGHT_BROWSER を True にして流し直してください")
            return None
        except Exception:
            logger.exception("Playwright での取得に失敗しました: %s", url)
            self._warn("Playwright でもページを取得できませんでした: %s", url)
            return None
        if not html:
            return None
        return FetchResult(url=url, html=html, source="Playwright", soup=parse_html(html))

    async def _render_async(self, url: str, wait_selector: str = "") -> str:
        # ブラウザ本体を入れていないと ModuleNotFoundError / Error になるので、
        # import はここで行い、呼び出し側で受け止める。
        from playwright.async_api import Error as PlaywrightError
        from playwright.async_api import TimeoutError as PlaywrightTimeoutError
        from playwright.async_api import async_playwright

        timeout_ms = int(self.network.playwright_timeout_sec * 1000)
        async with async_playwright() as playwright:
            browser = await playwright.chromium.launch(headless=True)
            try:
                page = await browser.new_page(user_agent=self.network.user_agent)
                try:
                    await page.goto(url, wait_until="domcontentloaded", timeout=timeout_ms)
                    if wait_selector:
                        await page.wait_for_selector(wait_selector, timeout=timeout_ms)
                except PlaywrightTimeoutError:
                    logger.warning("Playwright の待機がタイムアウトしました: %s", url)
                except PlaywrightError:
                    logger.exception("Playwright の遷移に失敗しました: %s", url)
                    return ""
                return await page.content()
            finally:
                await browser.close()

    def fetch_many(self, urls: Sequence[str], **kwargs: Any) -> Dict[str, Optional[FetchResult]]:
        """複数ページをまとめて取る。Konwenga の CBP 7ページ取得と同じ形。
        ホスト単位の間隔は HostThrottle が見るので、並列でも1サイトに集中しない。"""
        unique_urls = list(dict.fromkeys(url for url in urls if str(url or "").strip()))
        results: Dict[str, Optional[FetchResult]] = {}
        if not unique_urls:
            return results
        max_workers = min(self.network.max_workers, len(unique_urls))
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(self.fetch, url, **kwargs): url for url in unique_urls}
            for future in as_completed(futures):
                url = futures[future]
                try:
                    results[url] = future.result()
                except Exception:
                    logger.exception("ページ取得中に想定外のエラー: %s", url)
                    results[url] = None
        return results

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [4/10] 設定② 試合情報と記事設定
# ------------------------------------------------------------
# このセルを実行すると、Jリーグ公式の日程検索から [2/10] で指定した
# 2クラブの試合を探し、試合ページの内容を入力欄に入れて表示する。
#
# 入力欄の下に「入力チェック」が出る。いま何が埋まっていて、
# 何を自分で書く必要があるかが一覧で分かる。直したら
# 「もう一度チェック」ボタンを押せば更新される。
#
#   ✅ 自動   Jリーグ公式から入った
#   ✅ 入力済 自分で入れた
#   ✏️ 要入力 ここを埋めないと記事が成り立たない
#   －  任意   空欄のままでもよい
#
# 記事番号と両チームの布陣は、毎回変わるのに自動では決められない。
# 前回の値が残っていると直し忘れるので、空欄から始まるようにしてある。
#
# 日程一覧はカテゴリ別（J1/J2/J3）に引く。自チームの所属から自動で選ぶ。
# ただし一覧に出るのは今週ぶん（おおよそ 前2日〜後5日）だけ。
# もっと先の試合を扱うときは、試合ページのURLを MATCH_PAGE_URL に
# 直接貼れば、日程検索を飛ばしてそこだけ読みに行く。
# ============================================================

# --- 自動取得 ---------------------------------------------------
AUTO_FILL_FROM_JLEAGUE = True  # @param {type:"boolean"}
# 空欄なら下の JLEAGUE_SCHEDULE_URL から探す。
# 例: https://www.jleague.jp/match/j1/2026/091902/
MATCH_PAGE_URL = ""  # @param {type:"string"}

# --- 記事の識別 -------------------------------------------------
# 毎回同じ値を使うものだけ @param に残している。
SEASON = "2026/27"  # @param {type:"string"}
# 空欄なら自チームの所属カテゴリから「明治安田J1リーグ」を組み立てる。
COMPETITION = ""  # @param {type:"string"}

# --- 対戦カードの体裁 -------------------------------------------
USE_OFFICIAL_CLUB_NAME = True  # @param {type:"boolean"}
# 空欄ならクラブ情報のハッシュタグから自動で選ぶ。
OPPONENT_HASHTAG = ""  # @param {type:"string"}

# 記事番号・布陣・節・ホーム/アウェイ・キックオフ・会場・中継・天気は、
# このセルの下に出る入力欄で編集する。Colab の @param はセルのソースと
# 結びついていて、実行時に取得した値を表示へ反映できないため。

# --- 章立ての細かさ ---------------------------------------------
ATTACK_POINT_COUNT = 2  # @param {type:"slider", min:1, max:5, step:1}
DEFENSE_POINT_COUNT = 2  # @param {type:"slider", min:1, max:5, step:1}
INCLUDE_REFERENCE_SECTION = True  # @param {type:"boolean"}

# --- クラブ公式サイトの収集 ---------------------------------------
SCRAPE_CLUB_SITE = False  # @param {type:"boolean"}
USE_PLAYWRIGHT_FALLBACK = False  # @param {type:"boolean"}
RESPECT_ROBOTS_TXT = True  # @param {type:"boolean"}
REQUEST_MIN_INTERVAL_SEC = 1.0  # @param {type:"number"}

# --- 出力先 ----------------------------------------------------
DRIVE_FOLDER_NAME = "グラぽ マッチプレビュー"  # @param {type:"string"}

# --- ログ ------------------------------------------------------
LOG_LEVEL = "INFO"  # @param ["INFO", "DEBUG", "WARNING"]

setup_logging(LOG_LEVEL)

# Jリーグ公式の日程検索。自チームのカテゴリで出し分ける。
# リーグ戦・ルヴァンカップ・天皇杯をまとめて拾う。
# 表示されるのは今週ぶん（おおよそ 前2日〜後5日）で、期間指定のパラメータは効かない。
# 先の試合を扱うときは MATCH_PAGE_URL に試合ページのURLを貼る。
JLEAGUE_SCHEDULE_URL_TEMPLATE: Final[str] = (
    "https://www.jleague.jp/{league}/match/search-list/"
    "?category={league}%2Cleaguecup%2Cemperor"
)
JLEAGUE_SCHEDULE_LEAGUES: Final[Tuple[str, ...]] = ("j1", "j2", "j3")


def jleague_schedule_url(league: str) -> str:
    """"J1" / "j1" のどちらでも受ける。"""
    code = str(league or "j1").strip().lower()
    if code not in JLEAGUE_SCHEDULE_LEAGUES:
        logger.warning("知らないカテゴリです: %r。J1として扱います", league)
        code = "j1"
    return JLEAGUE_SCHEDULE_URL_TEMPLATE.format(league=code)


def search_league_hint(team: str) -> Optional[str]:
    """自チームの所属カテゴリ。日程検索のURLを決めるために使う。

    このセルはクラブ情報 [5/10] より前に走るので、ここでは分からないと返す。
    [6/10] が読み込まれると、クラブ情報から答える版で上書きされる。
    分からない場合は J1 → J2 → J3 の順に当たる。"""
    return None


# ------------------------------------------------------------
# 固定の文言とスタイル
# ------------------------------------------------------------
WEEKDAY_JA: Final[Tuple[str, ...]] = ("月曜日", "火曜日", "水曜日", "木曜日", "金曜日", "土曜日", "日曜日")


class SectionLabel:
    """記事の骨格に出てくる絵文字つき見出し。表記ゆれを避けるため定数化する。"""

    SPEAKER_DANKOBA: Final[str] = "【🎙 ダンコバ】"
    SPEAKER_ATTACK: Final[str] = "【⚔️ 攻撃担当】"
    SPEAKER_DEFENSE: Final[str] = "【🛡 守備担当】"
    GUIDE: Final[str] = "📌 試合観戦ガイド＆インフォメーション"
    GOODS: Final[str] = "🍡 グッズ・イベント・アクセス情報"
    SITUATION: Final[str] = "⚽️ 両チームの状況と先発予想"
    WIN_PATH: Final[str] = "🔥 {my_team}の勝ち筋"
    ATTACK: Final[str] = "⚔️ 攻撃のポイント"
    DEFENSE: Final[str] = "🛡 守備のポイント"
    CLOSING: Final[str] = "📝 おわりに"
    KICKOFF: Final[str] = "⏰ キックオフ"
    VENUE: Final[str] = "🏟 試合会場"
    BROADCAST: Final[str] = "📺 試合中継"
    WEATHER: Final[str] = "☀️ 天気予報"
    REFERENCE: Final[str] = "🔗 参考リンク（公開前に削除してください）"


@dataclass(frozen=True)
class DocsStyleConfig:
    google_api_max_retries: int = 6
    google_api_initial_wait_sec: float = 1.0
    google_api_max_wait_sec: float = 16.0
    google_api_request_interval_sec: float = 0.05

    # 表のレイアウト
    table_font_pt: float = 11.0
    table_cell_padding_pt: float = 12.0
    table_min_column_width_pt: float = 26.0
    table_max_total_width_pt: float = 450.0
    table_align_center: bool = True

    # 表のチーム名セル
    my_team_cell_bg: str = "#d80c18"
    my_team_cell_text: str = "#ffffff"
    opponent_team_cell_bg: str = "#cfe8fa"

    # 後から書き換えるプレースホルダの見た目
    placeholder_color: str = "#808080"
    placeholder_italic: bool = True


def normalize_serial_number(value: Any) -> str:
    """"286" / "D286" / "d 286" を "286" に揃える。"""
    text = str(value or "").strip().upper()
    text = re.sub(r"^D\s*", "", text)
    digits = re.sub(r"\D", "", text)
    if not digits:
        logger.warning("通し番号 %r から数字を読み取れませんでした。タイトルは 'D' のみになります", value)
    return digits


def parse_kickoff_date(value: Any) -> Optional[date]:
    text = str(value or "").strip()
    if not text:
        logger.warning("キックオフ日が空です。タイトルの日付はプレースホルダになります")
        return None
    for pattern in ("%Y-%m-%d", "%Y/%m/%d", "%Y.%m.%d"):
        try:
            return datetime.strptime(text, pattern).date()
        except ValueError:
            continue
    logger.warning("キックオフ日 %r を解釈できませんでした。YYYY-MM-DD で入力してください", value)
    return None


# ============================================================
# Jリーグ公式の日程・試合ページから初期値を拾う
# ============================================================
@dataclass(frozen=True)
class JLeagueMatchInfo:
    """試合ページから読み取れた値。読めなかった項目は空のまま。"""

    url: str = ""
    kickoff_date: str = ""      # YYYY-MM-DD
    kickoff_time: str = ""      # HH:MM
    section_no: str = ""
    home_team: str = ""
    away_team: str = ""
    venue_name: str = ""
    venue_address: str = ""
    venue_map_url: str = ""
    broadcast: str = ""

    def as_rows(self) -> List[Tuple[str, str]]:
        return [
            ("試合ページ", self.url),
            ("キックオフ日", self.kickoff_date),
            ("キックオフ時刻", self.kickoff_time),
            ("節", self.section_no),
            ("ホーム", self.home_team),
            ("アウェイ", self.away_team),
            ("会場", self.venue_name),
            ("住所", self.venue_address),
            ("地図", self.venue_map_url),
            ("中継", self.broadcast),
        ]


def search_name_variants(name: str) -> List[str]:
    """日程一覧との突き合わせに使うクラブ名の候補。

    このセルはクラブ情報 [5/10] より前に走るので、ここでは入力をそのまま返す。
    [6/10] が読み込まれると、正式名称と短縮名も足す版で上書きされる。
    「グランパス」のような愛称で入力した場合は [10/10] が取り直す。"""
    text = str(name or "").strip()
    return [text] if text else []


class JLeagueScheduleLookup:
    """日程検索から該当カードの試合ページを探し、その中身を読む。

    試合ページはサーバ側で組み立てられているので静的HTMLで読める。
    日程検索のほうはJavaScriptで描いている可能性があるので、
    リンクが1本も見つからなければ Playwright に落とす。"""

    # /match/j1/2026/091902/ の形
    MATCH_URL_PATTERN = re.compile(
        r"/match/(j1|j2|j3|leaguecup|emperor|acle|acl2|acl)/(20\d{2})/(\d{6})/?"
    )
    DATE_PATTERN = re.compile(r"(20\d{2})\s*/\s*(\d{1,2})\s*/\s*(\d{1,2})")
    SECTION_PATTERN = re.compile(r"第\s*(\d{1,3})\s*節")
    KICKOFF_PATTERN = re.compile(r"KICK\s*OFF\s*(\d{1,2})\s*[:：]\s*(\d{2})", re.IGNORECASE)
    ADDRESS_PATTERN = re.compile(r"住所\s*([^\s].{2,60}?)\s*(?:地図|https?://)")
    # 中継はページヘッダの o-page-header__broadcast の中だけを見る。
    # 全文から探すと、フッタの「J.LEAGUE OFFICIAL BROADCASTING PARTNER」の
    # DAZNロゴを毎回拾ってしまう。
    BROADCAST_CONTAINER_CLASS: Final[str] = "o-page-header__broadcast"
    BROADCAST_ITEM_CLASS: Final[str] = "o-page-header__broadcast-item"
    # クラス名が変わっても拾えるよう、部分一致でも探す
    BROADCAST_CLASS_HINT: Final[str] = "broadcast"
    BROADCAST_ITEM_CLASS_HINT: Final[str] = "broadcast-item"
    DAZN_HOST: Final[str] = "dazn.com"

    def __init__(
        self,
        fetcher: PageFetcher,
        report: Optional[RunReport] = None,
        render_on_miss: bool = False,
    ):
        self.fetcher = fetcher
        self.report = report
        # 静的HTMLに中継欄が無かったとき、ブラウザで描画し直すか
        self.render_on_miss = render_on_miss
        # 日程一覧で当たった行。試合ページから中継が取れなかったときの控え
        self.last_schedule_row: Optional[Any] = None

    def _warn(self, message: str, *args: Any) -> None:
        emit_warning(self.report, message, *args)

    # ---- 日程検索から試合ページを探す ----------------------------
    @staticmethod
    def _loose_match(listed_name: str, wanted_name: str) -> bool:
        """一覧の表記（「名古屋」「ＦＣ東京」）と入力（「名古屋グランパス」）を
        ゆるく突き合わせる。短いほうが長いほうに含まれていれば同じとみなす。"""
        listed = normalize_club_key(listed_name)
        wanted = normalize_club_key(wanted_name)
        if not listed or not wanted:
            return False
        return listed in wanted or wanted in listed

    def _container_element(self, anchor, max_levels: int = 6):
        """_container_text と同じたどり方で、テキストではなく要素を返す。
        日程一覧の行には中継の表示も入っているので、控えとして使う。"""
        node = anchor
        best = anchor
        for _ in range(max_levels):
            node = node.parent
            if node is None or node.name in ("body", "html", "[document]"):
                break
            urls = {
                match.group(0) for link in node.find_all("a", href=True)
                if (match := self.MATCH_URL_PATTERN.search(str(link.get("href"))))
            }
            if len(urls) != 1:
                break
            best = node
        return best

    def _container_text(self, anchor, max_levels: int = 6) -> str:
        """リンク1本ぶんの「行」のテキスト。

        文字数で打ち切ると、一覧全体を1つの塊として拾ってしまい
        どの候補も両チーム名を含むことになる。そこで、祖先に含まれる
        試合リンクが1本のままである限り上へたどり、2本目が現れる
        直前で止める。マークアップの形に依存しない。"""
        best = anchor.get_text(" ", strip=True)
        node = anchor
        for _ in range(max_levels):
            node = node.parent
            if node is None or node.name in ("body", "html", "[document]"):
                break
            # 1つの行に「対戦データ」「空リンク」など同じ試合への
            # リンクが複数あるので、本数ではなく URL の種類で数える。
            urls = {
                match.group(0) for link in node.find_all("a", href=True)
                if (match := self.MATCH_URL_PATTERN.search(str(link.get("href"))))
            }
            if len(urls) != 1:
                break
            best = node.get_text(" ", strip=True)
        return best

    def find_match_url_in(self, my_team: str, opponent_team: str, schedule_url: str) -> str:
        """1つの日程一覧から探す。見つからなければ空文字。"""
        page = self.fetcher.fetch(
            schedule_url,
            use_playwright_fallback=True,
            wait_selector="a[href*='/match/']",
        )
        if page is None or page.soup is None:
            logger.info("日程検索を取得できませんでした: %s", schedule_url)
            return ""

        candidates: Dict[str, str] = {}
        for anchor in page.soup.find_all("a", href=True):
            href = str(anchor.get("href"))
            if not self.MATCH_URL_PATTERN.search(href):
                continue
            url = urljoin(page.url, href)
            if url in candidates:
                continue
            surrounding = self._container_text(anchor)
            if self._loose_match_pair(surrounding, my_team, opponent_team):
                candidates[url] = surrounding
                if self.last_schedule_row is None:
                    self.last_schedule_row = self._container_element(anchor)

        if not candidates:
            return ""
        if len(candidates) > 1:
            logger.info("候補が %s件 見つかりました。先頭を使います", len(candidates))
            for url, text in candidates.items():
                logger.debug("  候補: %s / %s", url, text[:60])
        return next(iter(candidates))

    def find_match_url(self, my_team: str, opponent_team: str) -> str:
        """自チームのカテゴリの日程一覧から探す。
        カテゴリが分からないときは J1 → J2 → J3 の順に当たる。"""
        hint = search_league_hint(my_team)
        leagues = (hint.lower(),) if hint else JLEAGUE_SCHEDULE_LEAGUES
        for league in leagues:
            url = self.find_match_url_in(my_team, opponent_team, jleague_schedule_url(league))
            if url:
                logger.info("%s の日程一覧から見つけました", league.upper())
                return url
        self._warn(
            "日程検索から %s vs %s の試合を見つけられませんでした。"
            "一覧に出るのは今週ぶんだけなので、先の試合なら MATCH_PAGE_URL に"
            "試合ページのURLを貼ってください",
            my_team, opponent_team,
        )
        return ""

    def _loose_match_pair(self, text: str, my_team: str, opponent_team: str) -> bool:
        """1行ぶんのテキストに両チームが出ているか。
        入力が正式名称でも短縮名でも通るよう、部分一致の両方向で見る。"""
        normalized = normalize_club_key(text)
        fragments = self._name_fragments(text)

        def is_present(team: str) -> bool:
            for variant in search_name_variants(team):
                variant_key = normalize_club_key(variant)
                if not variant_key:
                    continue
                if variant_key in normalized:
                    return True
                if any(self._loose_match(fragment, variant) for fragment in fragments):
                    return True
            return False

        return is_present(my_team) and is_present(opponent_team)

    @staticmethod
    def _name_fragments(text: str) -> List[str]:
        """空白区切りの断片。一覧の「名古屋」「ＦＣ東京」を個別に取り出す。"""
        return [fragment for fragment in re.split(r"[\s　]+", text) if len(fragment) >= 2]

    # ---- 試合ページを読む --------------------------------------
    def read_match_page(self, match_url: str) -> Optional[JLeagueMatchInfo]:
        page = self.fetcher.fetch(match_url)
        if page is None or page.soup is None:
            self._warn("試合ページを取得できませんでした: %s", match_url)
            return None

        soup = page.soup
        text = soup.get_text(" ", strip=True)

        kickoff_date = ""
        if date_match := self.DATE_PATTERN.search(text):
            year, month, day = (int(value) for value in date_match.groups())
            try:
                kickoff_date = date(year, month, day).isoformat()
            except ValueError:
                logger.debug("日付として解釈できませんでした: %s", date_match.group(0))

        kickoff_time = ""
        if time_match := self.KICKOFF_PATTERN.search(text):
            kickoff_time = f"{int(time_match.group(1)):02d}:{time_match.group(2)}"

        section_no = ""
        if section_match := self.SECTION_PATTERN.search(text):
            section_no = section_match.group(1)

        home_team, away_team = self._extract_teams(soup, text)
        venue_name, venue_address, venue_map_url = self._extract_venue(soup, text)

        info = JLeagueMatchInfo(
            url=page.url,
            kickoff_date=kickoff_date,
            kickoff_time=kickoff_time,
            section_no=section_no,
            home_team=home_team,
            away_team=away_team,
            venue_name=venue_name,
            venue_address=venue_address,
            venue_map_url=venue_map_url,
            broadcast=self._resolve_broadcast(soup, page.url),
        )
        missing = [label for label, value in info.as_rows() if not value]
        if missing:
            logger.info("試合ページから読めなかった項目: %s", missing)
        return info

    @staticmethod
    def _dedupe_repeated_name(name: str) -> str:
        """クラブ名のリンクは「ＦＣ東京FC東京」「名古屋グランパス名古屋」のように、
        正式名称のうしろに短縮名がくっつく。短縮名は正式名称の先頭部分なので、
        後半が前半の接頭辞になる切れ目を長いほうから探して前半を採る。"""
        for split_at in range(len(name) - 1, 0, -1):
            head, tail = name[:split_at], name[split_at:]
            head_key, tail_key = normalize_club_key(head), normalize_club_key(tail)
            if tail_key and head_key.startswith(tail_key):
                return head
        return name

    @classmethod
    def _extract_teams(cls, soup: BeautifulSoup, text: str) -> Tuple[str, str]:
        """ホームが先、アウェイが後。クラブページへのリンクの並びで判断する。
        タイトルの「A vs B」より、リンクのほうが表記が安定している。"""
        club_names: List[str] = []
        for anchor in soup.find_all("a", href=True):
            if not re.search(r"/club/[a-z0-9]+/?$", str(anchor.get("href"))):
                continue
            name = anchor.get_text(" ", strip=True)
            # 「5位(4勝2分1敗) ＦＣ東京FC東京5位(4勝2分1敗)」のように順位が混ざる
            name = re.sub(r"\d+位\s*[（(][^）)]*[）)]", " ", name).strip()
            fragments = [part for part in re.split(r"[\s　]+", name) if part]
            if fragments:
                club_names.append(cls._dedupe_repeated_name(fragments[0]))
            if len(club_names) >= 2:
                break
        if len(club_names) >= 2:
            return club_names[0], club_names[1]
        if title_match := re.search(r"([^\s|【】]+)\s*vs\s*([^\s|【】]+)", text):
            return title_match.group(1), title_match.group(2)
        return "", ""

    @classmethod
    def _extract_venue(cls, soup: BeautifulSoup, text: str) -> Tuple[str, str, str]:
        """会場名と住所は、ページに貼られているGoogleマップのリンクから取る。
        query が「<会場名> 日本 <住所>」の形になっているので、そこを割る。"""
        venue_name = ""
        venue_address = ""
        venue_map_url = ""
        for anchor in soup.find_all("a", href=True):
            href = str(anchor.get("href"))
            if "google.com/maps" not in href or "query=" not in href:
                continue
            query = parse_qs(urlparse(href).query).get("query", [""])[0]
            if not query:
                continue
            venue_map_url = href
            parts = re.split(r"\s+日本\s+", query, maxsplit=1)
            venue_name = parts[0].strip()
            if len(parts) > 1:
                venue_address = parts[1].strip()
            break
        if not venue_address and (address_match := cls.ADDRESS_PATTERN.search(text)):
            venue_address = address_match.group(1).strip()
        return venue_name, venue_address, venue_map_url

    @staticmethod
    def _class_list(node) -> List[str]:
        return [str(name) for name in (node.get("class") or [])]

    @classmethod
    def _broadcast_container(cls, root) -> Optional[Any]:
        """中継欄の入れ物を探す。まず指定のクラス、だめならクラス名に
        broadcast を含む要素。フッタの放送パートナー欄は除く。"""
        container = root.find(class_=cls.BROADCAST_CONTAINER_CLASS)
        if container is not None:
            return container
        for node in root.find_all(attrs={"class": True}):
            classes = cls._class_list(node)
            if not any(cls.BROADCAST_CLASS_HINT in name.lower() for name in classes):
                continue
            if any(cls.BROADCAST_ITEM_CLASS_HINT in name.lower() for name in classes):
                node = node.parent if node.parent is not None else node
            if node.find_parent("footer") is not None:
                continue
            if any("partner" in name.lower() for name in cls._class_list(node)):
                continue
            logger.info("中継欄をクラス名の部分一致で見つけました: %s", " ".join(cls._class_list(node)))
            return node
        return None

    @classmethod
    def _read_broadcast_items(cls, root) -> str:
        """入れ物の中から放送局名を並べる。
          DAZN            … <a> の中の <img alt="DAZN">
          NHK BS など     … <span> のテキスト
        日程一覧では「スカチャン5・スカパー！動画ストア」のように
        1つの要素へ「・」で連結されることがあるので割る。"""
        items = root.find_all(class_=cls.BROADCAST_ITEM_CLASS)
        if not items:
            items = [
                node for node in root.find_all(attrs={"class": True})
                if any(cls.BROADCAST_ITEM_CLASS_HINT in name.lower() for name in cls._class_list(node))
            ]
        names: List[str] = []
        for item in items:
            image = item.find("img")
            name = str(image.get("alt") or "").strip() if image is not None else ""
            if not name:
                name = item.get_text(" ", strip=True)
            names.extend(part.strip() for part in re.split(r"[・/／]", name) if part.strip())
        if not names:
            # 項目要素が無くても、DAZNへのリンクがあればDAZN中継とみなす
            for anchor in root.find_all("a", href=True):
                if cls.DAZN_HOST not in str(anchor.get("href")).lower():
                    continue
                image = anchor.find("img")
                names.append(str(image.get("alt") or "").strip() if image is not None else "DAZN")
        return " / ".join(dict.fromkeys(name for name in names if name))

    @classmethod
    def _extract_broadcast(cls, soup: BeautifulSoup) -> str:
        container = cls._broadcast_container(soup)
        if container is None:
            cls._log_broadcast_diagnostics(soup)
            return ""
        return cls._read_broadcast_items(container)

    @classmethod
    def _log_broadcast_diagnostics(cls, soup: BeautifulSoup) -> None:
        """なぜ取れなかったのかを切り分けられるように残す。
        クラス名が変わったのか、そもそもHTMLに入っていないのかで対処が違う。"""
        found = sorted({
            " ".join(cls._class_list(node)) for node in soup.find_all(attrs={"class": True})
            if any(cls.BROADCAST_CLASS_HINT in name.lower() for name in cls._class_list(node))
        })
        if found:
            logger.warning(
                "中継欄のクラス名が想定と違います。ページにあったクラス: %s", found[:5]
            )
        else:
            logger.warning(
                "中継欄が静的HTMLに含まれていません（JavaScriptで描画されている可能性）。"
                "[1/10] の INSTALL_PLAYWRIGHT_BROWSER と [4/10] の USE_PLAYWRIGHT_FALLBACK を"
                " True にして実行し直すと取れることがあります"
            )

    def _resolve_broadcast(self, soup: BeautifulSoup, match_url: str) -> str:
        """試合ページ → 日程一覧の行 → ブラウザ描画 の順に中継を探す。"""
        broadcast = self._extract_broadcast(soup)
        if broadcast:
            return broadcast

        if self.last_schedule_row is not None:
            broadcast = self._read_broadcast_items(self.last_schedule_row)
            if broadcast:
                logger.info("中継は日程一覧の行から拾いました: %s", broadcast)
                return broadcast

        if self.render_on_miss:
            logger.info("中継欄を探すため、試合ページをブラウザで描画し直します")
            rendered = self.fetcher.fetch(
                match_url, force_render=True,
                wait_selector=f".{self.BROADCAST_ITEM_CLASS}",
            )
            if rendered is not None and rendered.soup is not None:
                broadcast = self._extract_broadcast(rendered.soup)
                if broadcast:
                    logger.info("中継はブラウザ描画で拾えました: %s", broadcast)
                    return broadcast
        return ""

    # ---- まとめ ------------------------------------------------
    def resolve(
        self,
        my_team: str,
        opponent_team: str,
        match_page_url: str = "",
    ) -> Optional[JLeagueMatchInfo]:
        url = str(match_page_url or "").strip()
        if url:
            logger.info("指定された試合ページを読みます: %s", url)
        else:
            url = self.find_match_url(my_team, opponent_team)
            if not url:
                return None
            logger.info("日程検索から試合ページを見つけました: %s", url)
        return self.read_match_page(url)


# ============================================================
# 入力欄と入力チェック
# ------------------------------------------------------------
# Colab の @param は、表示している値がセルのソースコードと結びついている。
# 実行時に Python 側で代入しても表示は変わらないので、
# 「検索結果が入った状態で見える」必要がある項目はここに集めた。
#
# あわせて、何を自分で書く必要があるかを一覧で出す。
# 埋まっているのか、これから書くのかがフォームを見て分からない、
# というのが @param のいちばん困るところだった。
# ============================================================
try:
    import ipywidgets as widgets
    from IPython.display import display as _display
    _HAS_WIDGETS = True
except Exception:  # pragma: no cover - Colab以外で動かすとき
    _HAS_WIDGETS = False


class FieldKind(str, Enum):
    AUTO = "auto"          # Jリーグ公式から入る。入らなければ自分で書く
    REQUIRED = "required"  # 自動では決まらない。必ず自分で書く
    OPTIONAL = "optional"  # 空欄のままでもよい


@dataclass(frozen=True)
class FieldSpec:
    key: str
    label: str
    hint: str
    kind: FieldKind
    choices: Tuple[str, ...] = ()


SETTINGS_FIELDS: Final[Tuple[FieldSpec, ...]] = (
    # --- Jリーグ公式から入る ---
    FieldSpec("section_no", "節", "例: 8", FieldKind.AUTO),
    FieldSpec("home_or_away", "ホーム/アウェイ", "", FieldKind.AUTO, ("", "ホーム", "アウェイ")),
    FieldSpec("kickoff_date", "キックオフ日", "YYYY-MM-DD", FieldKind.AUTO),
    FieldSpec("kickoff_time", "キックオフ時刻", "HH:MM", FieldKind.AUTO),
    FieldSpec("venue_name", "会場", "例: ＭＵＦＧスタジアム", FieldKind.AUTO),
    FieldSpec("venue_address", "住所", "", FieldKind.AUTO),
    FieldSpec("venue_map_url", "地図URL", "Googleマップ", FieldKind.AUTO),
    FieldSpec("broadcast", "中継", "例: DAZN / NHK BS", FieldKind.AUTO),
    # --- 自動では決まらない ---
    FieldSpec("serial_number", "記事番号", "例: 288（D は不要）", FieldKind.REQUIRED),
    FieldSpec("my_team_formation", "自チーム布陣", "例: 3-4-2-1", FieldKind.REQUIRED),
    FieldSpec("opponent_formation", "相手布陣", "例: 4-3-3", FieldKind.REQUIRED),
    # --- 任意 ---
    FieldSpec("weather_text", "天気", "例: 晴 / 気温 27℃前後の予想です。", FieldKind.OPTIONAL),
    FieldSpec("weather_url", "天気URL", "tenki.jp など", FieldKind.OPTIONAL),
)

FIELD_GROUPS: Final[Tuple[Tuple[str, FieldKind], ...]] = (
    ("Jリーグ公式から取得（違っていれば直してください）", FieldKind.AUTO),
    ("自分で入力（毎回変わるので空欄から始まります）", FieldKind.REQUIRED),
    ("任意（空欄でも記事は作れます）", FieldKind.OPTIONAL),
)

_STATUS_STYLES: Final[Dict[str, Tuple[str, str]]] = {
    "auto": ("✅ 自動", "#1a7f37"),
    "typed": ("✅ 入力済", "#1a7f37"),
    "missing": ("✏️ 要入力", "#b3261e"),
    "blank_optional": ("－ 任意", "#8a8a8a"),
}


class MatchSettingsForm:
    """試合情報と記事設定の入力欄。取得できた値を初期値として持ち、
    何が埋まっていて何を書く必要があるかを一覧で出す。"""

    def __init__(self) -> None:
        self._specs = {spec.key: spec for spec in SETTINGS_FIELDS}
        self._values: Dict[str, str] = {spec.key: "" for spec in SETTINGS_FIELDS}
        self._auto_filled: set = set()
        self._widgets: Dict[str, Any] = {}
        self._status: Any = None
        self._checklist: Any = None
        self.source_url: str = ""
        if _HAS_WIDGETS:
            self._build_widgets()

    # ---- 組み立て ----------------------------------------------
    def _build_widgets(self) -> None:
        style = {"description_width": "120px"}
        layout = widgets.Layout(width="560px")
        for spec in SETTINGS_FIELDS:
            if spec.choices:
                self._widgets[spec.key] = widgets.Dropdown(
                    options=list(spec.choices), value="",
                    description=spec.label, style=style, layout=layout,
                )
            else:
                self._widgets[spec.key] = widgets.Text(
                    value="", description=spec.label, placeholder=spec.hint,
                    style=style, layout=layout,
                )
        self._status = widgets.HTML(value="")
        self._checklist = widgets.HTML(value="")

    def display(self) -> None:
        if not _HAS_WIDGETS:
            logger.warning(
                "ipywidgets が使えないため入力欄を出せません。"
                "SETTINGS.set(serial_number='288') のように書いてください"
            )
            self.log_checklist()
            return

        rows: List[Any] = [self._status]
        for title, kind in FIELD_GROUPS:
            rows.append(widgets.HTML(
                f'<div style="margin:10px 0 2px;font-weight:600;'
                f'border-bottom:1px solid #ddd">{title}</div>'
            ))
            rows.extend(self._widgets[spec.key] for spec in SETTINGS_FIELDS if spec.kind is kind)

        recheck = widgets.Button(description="もう一度チェック", icon="refresh")
        recheck.on_click(lambda _button: self.refresh_checklist())
        rows.append(widgets.HTML('<div style="margin-top:12px"></div>'))
        rows.append(recheck)
        rows.append(self._checklist)
        self.refresh_checklist()
        _display(widgets.VBox(rows))

    # ---- 値の出し入れ ------------------------------------------
    def values(self) -> Dict[str, str]:
        """いま入力欄に入っている値。実行時([10/10])に読む。"""
        if not _HAS_WIDGETS:
            return dict(self._values)
        return {key: str(widget.value or "").strip() for key, widget in self._widgets.items()}

    def set(self, **kwargs: str) -> None:
        for key, value in kwargs.items():
            if key not in self._specs:
                logger.warning("知らない項目です: %s", key)
                continue
            text = str(value or "")
            self._values[key] = text
            if _HAS_WIDGETS:
                self._widgets[key].value = text

    def apply_match_info(self, info: JLeagueMatchInfo, my_team: str) -> List[str]:
        """空欄の項目だけ埋める。手で直した値は上書きしない。"""
        current = self.values()
        filled: List[str] = []
        mapping = {
            "section_no": info.section_no,
            "kickoff_date": info.kickoff_date,
            "kickoff_time": info.kickoff_time,
            "venue_name": info.venue_name,
            "venue_address": info.venue_address,
            "venue_map_url": info.venue_map_url,
            "broadcast": info.broadcast,
        }
        if info.home_team:
            side = "ホーム" if JLeagueScheduleLookup._loose_match(info.home_team, my_team) else "アウェイ"
            mapping["home_or_away"] = side
        for key, value in mapping.items():
            if value and not current.get(key, "").strip():
                self.set(**{key: value})
                self._auto_filled.add(key)
                filled.append(self._specs[key].label)
        self.source_url = info.url
        return filled

    def set_status(self, message: str, ok: bool = True) -> None:
        color = "#1a7f37" if ok else "#b3261e"
        if _HAS_WIDGETS and self._status is not None:
            self._status.value = (
                f'<div style="color:{color};font-weight:600;padding:4px 0">{message}</div>'
            )
        else:
            logger.info(message)

    # ---- 入力チェック ------------------------------------------
    def _field_status(self, spec: FieldSpec, value: str) -> str:
        if value.strip():
            return "auto" if spec.key in self._auto_filled else "typed"
        return "blank_optional" if spec.kind is FieldKind.OPTIONAL else "missing"

    def missing_fields(self) -> List[str]:
        """埋めないと記事が成り立たない項目のうち、まだ空欄のもの。"""
        current = self.values()
        return [
            spec.label for spec in SETTINGS_FIELDS
            if self._field_status(spec, current.get(spec.key, "")) == "missing"
        ]

    def refresh_checklist(self) -> None:
        if not _HAS_WIDGETS or self._checklist is None:
            self.log_checklist()
            return
        current = self.values()
        rows = []
        for spec in SETTINGS_FIELDS:
            value = current.get(spec.key, "")
            status = self._field_status(spec, value)
            label, color = _STATUS_STYLES[status]
            shown = value if len(value) <= 48 else value[:45] + "…"
            rows.append(
                f'<tr><td style="padding:2px 12px 2px 0;color:{color};white-space:nowrap">{label}</td>'
                f'<td style="padding:2px 12px 2px 0;white-space:nowrap">{spec.label}</td>'
                f'<td style="padding:2px 0;color:#444">{shown or "&mdash;"}</td></tr>'
            )
        missing = self.missing_fields()
        if missing:
            summary = (
                f'<div style="color:#b3261e;font-weight:600;margin-bottom:4px">'
                f'要入力 {len(missing)}件: {"、".join(missing)}</div>'
            )
        else:
            summary = (
                '<div style="color:#1a7f37;font-weight:600;margin-bottom:4px">'
                '必要な項目はすべて埋まっています</div>'
            )
        self._checklist.value = (
            '<div style="margin-top:8px">'
            '<div style="font-weight:600;margin-bottom:4px">入力チェック</div>'
            f'{summary}<table style="font-size:13px">{"".join(rows)}</table></div>'
        )

    def log_checklist(self) -> None:
        """ログにも同じ内容を出す。入力欄が出せない環境と、実行時の記録用。"""
        current = self.values()
        width = max(display_width(spec.label) for spec in SETTINGS_FIELDS)
        lines = ["", "--- 入力チェック ---"]
        for spec in SETTINGS_FIELDS:
            value = current.get(spec.key, "")
            label, _ = _STATUS_STYLES[self._field_status(spec, value)]
            shown = value if len(value) <= 60 else value[:57] + "…"
            lines.append(f"  {label}  {pad_display(spec.label, width)}  {shown or '—'}")
        missing = self.missing_fields()
        lines.append(f"  要入力 {len(missing)}件: {'、'.join(missing)}" if missing
                     else "  必要な項目はすべて埋まっています")
        lines.append("--------------------")
        logger.info("\n".join(lines))


def fetch_match_info(my_team: str, opponent_team: str) -> Optional[JLeagueMatchInfo]:
    """Jリーグ公式から試合情報を取ってくる。取れなければ None。"""
    if not AUTO_FILL_FROM_JLEAGUE:
        logger.info("AUTO_FILL_FROM_JLEAGUE が False のため、自動取得は行いません")
        return None
    network = NetworkConfig(
        respect_robots=bool(RESPECT_ROBOTS_TXT),
        min_interval_sec=float(REQUEST_MIN_INTERVAL_SEC or 0),
    )
    lookup = JLeagueScheduleLookup(
        PageFetcher(build_session(network), network),
        render_on_miss=bool(USE_PLAYWRIGHT_FALLBACK),
    )
    return lookup.resolve(my_team, opponent_team, MATCH_PAGE_URL)


def populate_settings_form(
    form: MatchSettingsForm, my_team: str, opponent_team: str
) -> Optional[JLeagueMatchInfo]:
    info = fetch_match_info(my_team, opponent_team)
    if info is None:
        form.set_status(
            f"{my_team} vs {opponent_team} の試合をJリーグ公式から取得できませんでした。"
            "下の欄に手で入れるか、MATCH_PAGE_URL に試合ページのURLを貼ってください",
            ok=False,
        )
        form.refresh_checklist()
        return None
    filled = form.apply_match_info(info, my_team)
    message = f"Jリーグ公式から取得しました（{info.url}）"
    logger.info("入力欄に入れた項目: %s", "、".join(filled) if filled else "なし")
    form.set_status(message, ok=True)
    form.refresh_checklist()
    return info


SETTINGS = MatchSettingsForm()
JLEAGUE_MATCH_INFO = populate_settings_form(SETTINGS, MY_TEAM, OPPONENT_TEAM)
SETTINGS.display()

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [5/10] Jリーグクラブ情報
# ------------------------------------------------------------
# J1・J2・J3 全60クラブの正式名称をキーに、次を持つ。
#   league                … 所属カテゴリ（J1 / J2 / J3）
#   official_site_url     … クラブ公式サイト
#   jleague_profile_url   … Jリーグ公式のクラブプロフィール
#   football_lab_url      … Football LAB のチームデータ
#   official_x_account    … 公式Xアカウント（URL と handle）
#   official_x_hashtags   … 公式Xでクラブを表すハッシュタグ
#   aliases               … 短縮名・略称・愛称・独自の呼び方
#
# 昇降格やURL変更のときに触るのはこのセル。
# 呼び方を足したいだけなら、次のセルの GRAPO_EXTRA_ALIASES のほうが安全。
# ============================================================
from typing import Literal, TypedDict

League = Literal["J1", "J2", "J3"]
AliasCategory = Literal["short_name", "abbreviation", "nickname", "custom"]


class AliasNode(TypedDict):
    short_name: List[str]
    abbreviation: List[str]
    nickname: List[str]
    custom: List[str]


class OfficialXAccountNode(TypedDict):
    url: str
    handle: str


class XHashtagsNode(TypedDict):
    club: List[str]


class ClubNode(TypedDict):
    league: League
    official_site_url: str
    jleague_profile_url: str
    football_lab_url: str
    official_x_account: OfficialXAccountNode
    official_x_hashtags: XHashtagsNode
    aliases: AliasNode


# データを収集した日付。シーズン途中の昇降格やURL変更を追う目印にする。
DATA_AS_OF = "2026-09-17"

SOURCE_URLS = {
    "jleague_j1_clubs": "https://www.jleague.jp/j1/club/",
    "jleague_j2_clubs": "https://www.jleague.jp/j2/club/",
    "jleague_j3_clubs": "https://www.jleague.jp/j3/club/",
    "football_lab_team_select": "https://www.football-lab.jp/",
    "x": "https://x.com/",
}

# 正式名称（Jリーグ公式表記）をキーにしたクラブ情報。
# 各カテゴリ内はJリーグ公式クラブ一覧の並び順。
J_LEAGUE_CLUBS: Dict[str, ClubNode] = {
    # ------------------------------ J1 ------------------------------
    '鹿島アントラーズ': {
        'league': 'J1',
        'official_site_url': 'https://www.antlers.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kashima/',
        'football_lab_url': 'https://www.football-lab.jp/kasm/',
        'official_x_account': {'url': 'https://x.com/atlrs_official', 'handle': '@atlrs_official'},
        'official_x_hashtags': {'club': ['#antlers', '#kashima', '#鹿島アントラーズ']},
        'aliases': {'short_name': ['鹿島'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '水戸ホーリーホック': {
        'league': 'J1',
        'official_site_url': 'http://www.mito-hollyhock.net/',
        'jleague_profile_url': 'https://www.jleague.jp/club/mito/',
        'football_lab_url': 'https://www.football-lab.jp/mito/',
        'official_x_account': {'url': 'https://x.com/hollyhock_staff', 'handle': '@hollyhock_staff'},
        'official_x_hashtags': {'club': ['#水戸ホーリーホック', '#hollyhock']},
        'aliases': {'short_name': ['水戸'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '浦和レッズ': {
        'league': 'J1',
        'official_site_url': 'http://www.urawa-reds.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/urawa/',
        'football_lab_url': 'https://www.football-lab.jp/uraw/',
        'official_x_account': {'url': 'https://x.com/REDSOFFICIAL', 'handle': '@REDSOFFICIAL'},
        'official_x_hashtags': {'club': ['#浦和レッズ', '#urawareds', '#WeareREDS']},
        'aliases': {'short_name': ['浦和'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ジェフユナイテッド千葉': {
        'league': 'J1',
        'official_site_url': 'https://jefunited.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/chiba/',
        'football_lab_url': 'https://www.football-lab.jp/chib/',
        'official_x_account': {'url': 'https://x.com/jef_united', 'handle': '@jef_united'},
        'official_x_hashtags': {'club': ['#jefunited', '#ジェフ千葉']},
        'aliases': {'short_name': ['千葉'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '柏レイソル': {
        'league': 'J1',
        'official_site_url': 'https://www.reysol.co.jp/index.php/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kashiwa/',
        'football_lab_url': 'https://www.football-lab.jp/kasw/',
        'official_x_account': {'url': 'https://x.com/REYSOL_Official', 'handle': '@REYSOL_Official'},
        'official_x_hashtags': {'club': ['#柏レイソル', '#reysol', '#NoREYSOLNoLIFE']},
        'aliases': {'short_name': ['柏'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ＦＣ東京': {
        'league': 'J1',
        'official_site_url': 'https://www.fctokyo.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/ftokyo/',
        'football_lab_url': 'https://www.football-lab.jp/fctk/',
        'official_x_account': {'url': 'https://x.com/fctokyoofficial', 'handle': '@fctokyoofficial'},
        'official_x_hashtags': {'club': ['#fctokyo', '#tokyo', '#FC東京']},
        'aliases': {'short_name': ['FC東京'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '東京ヴェルディ': {
        'league': 'J1',
        'official_site_url': 'https://www.verdy.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/tokyov/',
        'football_lab_url': 'https://www.football-lab.jp/tk-v/',
        'official_x_account': {'url': 'https://x.com/TokyoVerdySTAFF', 'handle': '@TokyoVerdySTAFF'},
        'official_x_hashtags': {'club': ['#verdy', '#東京ヴェルディ', '#東京V']},
        'aliases': {'short_name': ['東京Ｖ'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ＦＣ町田ゼルビア': {
        'league': 'J1',
        'official_site_url': 'http://www.zelvia.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/machida/',
        'football_lab_url': 'https://www.football-lab.jp/mcd/',
        'official_x_account': {'url': 'https://x.com/FCMachidaZelvia', 'handle': '@FCMachidaZelvia'},
        'official_x_hashtags': {'club': ['#FC町田ゼルビア', '#zelvia']},
        'aliases': {'short_name': ['町田'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '川崎フロンターレ': {
        'league': 'J1',
        'official_site_url': 'https://www.frontale.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kawasakif/',
        'football_lab_url': 'https://www.football-lab.jp/ka-f/',
        'official_x_account': {'url': 'https://x.com/frontale_staff', 'handle': '@frontale_staff'},
        'official_x_hashtags': {'club': ['#frontale', '#川崎フロンターレ']},
        'aliases': {'short_name': ['川崎Ｆ'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '横浜Ｆ・マリノス': {
        'league': 'J1',
        'official_site_url': 'https://www.f-marinos.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/yokohamafm/',
        'football_lab_url': 'https://www.football-lab.jp/y-fm/',
        'official_x_account': {'url': 'https://x.com/prompt_fmarinos', 'handle': '@prompt_fmarinos'},
        'official_x_hashtags': {'club': ['#fmarinos', '#マリノスファミリー']},
        'aliases': {'short_name': ['横浜FM'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '清水エスパルス': {
        'league': 'J1',
        'official_site_url': 'https://www.s-pulse.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/shimizu/',
        'football_lab_url': 'https://www.football-lab.jp/shim/',
        'official_x_account': {'url': 'https://x.com/spulse_official', 'handle': '@spulse_official'},
        'official_x_hashtags': {'club': ['#spulse', '#エスパルス']},
        'aliases': {'short_name': ['清水'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '名古屋グランパス': {
        'league': 'J1',
        'official_site_url': 'http://nagoya-grampus.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/nagoya/',
        'football_lab_url': 'https://www.football-lab.jp/nago/',
        'official_x_account': {'url': 'https://x.com/nge_official', 'handle': '@nge_official'},
        'official_x_hashtags': {'club': ['#grampus', '#グランパス']},
        'aliases': {'short_name': ['名古屋'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '京都サンガF.C.': {
        'league': 'J1',
        'official_site_url': 'http://www.sanga-fc.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kyoto/',
        'football_lab_url': 'https://www.football-lab.jp/kyot/',
        'official_x_account': {'url': 'https://x.com/sangafc', 'handle': '@sangafc'},
        'official_x_hashtags': {'club': ['#京都サンガ', '#sanga', '#京都サンガFC']},
        'aliases': {'short_name': ['京都'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ガンバ大阪': {
        'league': 'J1',
        'official_site_url': 'http://www.gamba-osaka.net/',
        'jleague_profile_url': 'https://www.jleague.jp/club/gosaka/',
        'football_lab_url': 'https://www.football-lab.jp/g-os/',
        'official_x_account': {'url': 'https://x.com/GAMBA_OFFICIAL', 'handle': '@GAMBA_OFFICIAL'},
        'official_x_hashtags': {'club': ['#ガンバ大阪', '#GAMBAOSAKA']},
        'aliases': {'short_name': ['Ｇ大阪'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'セレッソ大阪': {
        'league': 'J1',
        'official_site_url': 'https://www.cerezo.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/cosaka/',
        'football_lab_url': 'https://www.football-lab.jp/c-os/',
        'official_x_account': {'url': 'https://x.com/crz_official', 'handle': '@crz_official'},
        'official_x_hashtags': {'club': ['#セレッソ大阪', '#cerezo']},
        'aliases': {'short_name': ['Ｃ大阪'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ヴィッセル神戸': {
        'league': 'J1',
        'official_site_url': 'https://www.vissel-kobe.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kobe/',
        'football_lab_url': 'https://www.football-lab.jp/kobe/',
        'official_x_account': {'url': 'https://x.com/visselkobe', 'handle': '@visselkobe'},
        'official_x_hashtags': {'club': ['#visselkobe', '#ヴィッセル神戸']},
        'aliases': {'short_name': ['神戸'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ファジアーノ岡山': {
        'league': 'J1',
        'official_site_url': 'http://www.fagiano-okayama.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/okayama/',
        'football_lab_url': 'https://www.football-lab.jp/okay/',
        'official_x_account': {'url': 'https://x.com/fagiano_koho', 'handle': '@fagiano_koho'},
        'official_x_hashtags': {'club': ['#ファジアーノ岡山']},
        'aliases': {'short_name': ['岡山'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'サンフレッチェ広島': {
        'league': 'J1',
        'official_site_url': 'http://www.sanfrecce.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/hiroshima/',
        'football_lab_url': 'https://www.football-lab.jp/hiro/',
        'official_x_account': {'url': 'https://x.com/sanfrecce_SFC', 'handle': '@sanfrecce_SFC'},
        'official_x_hashtags': {'club': ['#sanfrecce', '#サンフレッチェ広島']},
        'aliases': {'short_name': ['広島'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'アビスパ福岡': {
        'league': 'J1',
        'official_site_url': 'http://www.avispa.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/fukuoka/',
        'football_lab_url': 'https://www.football-lab.jp/fuku/',
        'official_x_account': {'url': 'https://x.com/AvispaF', 'handle': '@AvispaF'},
        'official_x_hashtags': {'club': ['#アビスパ福岡', '#avispa']},
        'aliases': {'short_name': ['福岡'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'Ｖ・ファーレン長崎': {
        'league': 'J1',
        'official_site_url': 'https://www.v-varen.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/nagasaki/',
        'football_lab_url': 'https://www.football-lab.jp/ngsk/',
        'official_x_account': {'url': 'https://x.com/v_varenstaff', 'handle': '@v_varenstaff'},
        'official_x_hashtags': {'club': ['#vvaren', '#Ｖ・ファーレン長崎']},
        'aliases': {'short_name': ['長崎'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    # ------------------------------ J2 ------------------------------
    '北海道コンサドーレ札幌': {
        'league': 'J2',
        'official_site_url': 'https://www.consadole-sapporo.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/sapporo/',
        'football_lab_url': 'https://www.football-lab.jp/sapp/',
        'official_x_account': {'url': 'https://x.com/consaofficial', 'handle': '@consaofficial'},
        'official_x_hashtags': {'club': ['#consadole', '#コンサドーレ', '#北海道コンサドーレ札幌']},
        'aliases': {'short_name': ['札幌'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ヴァンラーレ八戸': {
        'league': 'J2',
        'official_site_url': 'http://www.vanraure.net/',
        'jleague_profile_url': 'https://www.jleague.jp/club/hachinohe/',
        'football_lab_url': 'https://www.football-lab.jp/hach/',
        'official_x_account': {'url': 'https://x.com/vanraure', 'handle': '@vanraure'},
        'official_x_hashtags': {'club': ['#ヴァンラーレ八戸']},
        'aliases': {'short_name': ['八戸'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ベガルタ仙台': {
        'league': 'J2',
        'official_site_url': 'https://www.vegalta.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/sendai/',
        'football_lab_url': 'https://www.football-lab.jp/send/',
        'official_x_account': {'url': 'https://x.com/vega_official_', 'handle': '@vega_official_'},
        'official_x_hashtags': {'club': ['#VEGALTA', '#vegalta', '#ベガルタ仙台']},
        'aliases': {'short_name': ['仙台'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ブラウブリッツ秋田': {
        'league': 'J2',
        'official_site_url': 'http://blaublitz.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/akita/',
        'football_lab_url': 'https://www.football-lab.jp/aki/',
        'official_x_account': {'url': 'https://x.com/blaublitz_akita', 'handle': '@blaublitz_akita'},
        'official_x_hashtags': {'club': ['#ブラウブリッツ秋田', '#bbakita']},
        'aliases': {'short_name': ['秋田'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'モンテディオ山形': {
        'league': 'J2',
        'official_site_url': 'https://www.montedioyamagata.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/yamagata/',
        'football_lab_url': 'https://www.football-lab.jp/yama/',
        'official_x_account': {'url': 'https://x.com/monte_prstaff', 'handle': '@monte_prstaff'},
        'official_x_hashtags': {'club': ['#montedio', '#yamagataichigan']},
        'aliases': {'short_name': ['山形'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'いわきＦＣ': {
        'league': 'J2',
        'official_site_url': 'https://iwakifc.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/iwaki/',
        'football_lab_url': 'https://www.football-lab.jp/ifc/',
        'official_x_account': {'url': 'https://x.com/IwakiFcOfficial', 'handle': '@IwakiFcOfficial'},
        'official_x_hashtags': {'club': ['#iwakifc', '#いわきFC']},
        'aliases': {'short_name': ['いわき'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '栃木シティ': {
        'league': 'J2',
        'official_site_url': 'https://tochigi-city.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/tochigic/',
        'football_lab_url': 'https://www.football-lab.jp/to-c/',
        'official_x_account': {'url': 'https://x.com/tochigi_city_', 'handle': '@tochigi_city_'},
        'official_x_hashtags': {'club': ['#栃木シティ']},
        'aliases': {'short_name': ['栃木Ｃ'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ＲＢ大宮アルディージャ': {
        'league': 'J2',
        'official_site_url': 'https://www.ardija.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/omiya/',
        'football_lab_url': 'https://www.football-lab.jp/omiy/',
        'official_x_account': {'url': 'https://x.com/Ardija_Official', 'handle': '@Ardija_Official'},
        'official_x_hashtags': {'club': ['#RB大宮アルディージャ', '#ardija']},
        'aliases': {'short_name': ['大宮'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '横浜ＦＣ': {
        'league': 'J2',
        'official_site_url': 'https://www.yokohamafc.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/yokohamafc/',
        'football_lab_url': 'https://www.football-lab.jp/y-fc/',
        'official_x_account': {'url': 'https://x.com/yokohama_fc', 'handle': '@yokohama_fc'},
        'official_x_hashtags': {'club': ['#yokohamafc', '#横浜FC']},
        'aliases': {'short_name': ['横浜FC'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '湘南ベルマーレ': {
        'league': 'J2',
        'official_site_url': 'http://www.bellmare.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/shonan/',
        'football_lab_url': 'https://www.football-lab.jp/shon/',
        'official_x_account': {'url': 'https://x.com/bellmare_staff', 'handle': '@bellmare_staff'},
        'official_x_hashtags': {'club': ['#bellmare', '#ベルマーレ']},
        'aliases': {'short_name': ['湘南'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ヴァンフォーレ甲府': {
        'league': 'J2',
        'official_site_url': 'http://www.ventforet.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kofu/',
        'football_lab_url': 'https://www.football-lab.jp/kofu/',
        'official_x_account': {'url': 'https://x.com/vfk_official', 'handle': '@vfk_official'},
        'official_x_hashtags': {'club': ['#vfk', '#ヴァンフォーレ甲府']},
        'aliases': {'short_name': ['甲府'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'アルビレックス新潟': {
        'league': 'J2',
        'official_site_url': 'http://www.albirex.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/niigata/',
        'football_lab_url': 'https://www.football-lab.jp/niig/',
        'official_x_account': {'url': 'https://x.com/albirex_pr', 'handle': '@albirex_pr'},
        'official_x_hashtags': {'club': ['#albirex', '#アルビレックス新潟']},
        'aliases': {'short_name': ['新潟'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'カターレ富山': {
        'league': 'J2',
        'official_site_url': 'http://www.kataller.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/toyama/',
        'football_lab_url': 'https://www.football-lab.jp/toya/',
        'official_x_account': {'url': 'https://x.com/katallertoyama', 'handle': '@katallertoyama'},
        'official_x_hashtags': {'club': ['#カターレ富山', '#kataller']},
        'aliases': {'short_name': ['富山'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ジュビロ磐田': {
        'league': 'J2',
        'official_site_url': 'http://www.jubilo-iwata.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/iwata/',
        'football_lab_url': 'https://www.football-lab.jp/iwat/',
        'official_x_account': {'url': 'https://x.com/Jubiloiwata_YFC', 'handle': '@Jubiloiwata_YFC'},
        'official_x_hashtags': {'club': ['#ジュビロ磐田', '#jubilo']},
        'aliases': {'short_name': ['磐田'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '藤枝ＭＹＦＣ': {
        'league': 'J2',
        'official_site_url': 'http://myfc.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/fujieda/',
        'football_lab_url': 'https://www.football-lab.jp/fuji/',
        'official_x_account': {'url': 'https://x.com/fujiedamyfc_pr', 'handle': '@fujiedamyfc_pr'},
        'official_x_hashtags': {'club': ['#藤枝MYFC', '#fujiedamyfc']},
        'aliases': {'short_name': ['藤枝'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '徳島ヴォルティス': {
        'league': 'J2',
        'official_site_url': 'http://www.vortis.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/tokushima/',
        'football_lab_url': 'https://www.football-lab.jp/toku/',
        'official_x_account': {'url': 'https://x.com/vortis_pr', 'handle': '@vortis_pr'},
        'official_x_hashtags': {'club': ['#徳島ヴォルティス', '#vortis']},
        'aliases': {'short_name': ['徳島'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ＦＣ今治': {
        'league': 'J2',
        'official_site_url': 'http://www.fcimabari.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/imabari/',
        'football_lab_url': 'https://www.football-lab.jp/imab/',
        'official_x_account': {'url': 'https://x.com/FCimabari', 'handle': '@FCimabari'},
        'official_x_hashtags': {'club': ['#FC今治', '#fcimabari']},
        'aliases': {'short_name': ['今治'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'サガン鳥栖': {
        'league': 'J2',
        'official_site_url': 'http://www.sagan-tosu.net/',
        'jleague_profile_url': 'https://www.jleague.jp/club/tosu/',
        'football_lab_url': 'https://www.football-lab.jp/tosu/',
        'official_x_account': {'url': 'https://x.com/saganofficial17', 'handle': '@saganofficial17'},
        'official_x_hashtags': {'club': ['#サガン鳥栖', '#SAGANTOSU', '#sagantosu']},
        'aliases': {'short_name': ['鳥栖'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '大分トリニータ': {
        'league': 'J2',
        'official_site_url': 'https://www.oita-trinita.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/oita/',
        'football_lab_url': 'https://www.football-lab.jp/oita/',
        'official_x_account': {'url': 'https://x.com/TRINITAofficial', 'handle': '@TRINITAofficial'},
        'official_x_hashtags': {'club': ['#大分トリニータ', '#trinita']},
        'aliases': {'short_name': ['大分'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'テゲバジャーロ宮崎': {
        'league': 'J2',
        'official_site_url': 'https://www.tegevajaro.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/miyazaki/',
        'football_lab_url': 'https://www.football-lab.jp/myzk/',
        'official_x_account': {'url': 'https://x.com/55tegevajaro', 'handle': '@55tegevajaro'},
        'official_x_hashtags': {'club': ['#テゲバジャーロ宮崎', '#テゲバ']},
        'aliases': {'short_name': ['宮崎'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    # ------------------------------ J3 ------------------------------
    '福島ユナイテッドＦＣ': {
        'league': 'J3',
        'official_site_url': 'http://fufc.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/fukushima/',
        'football_lab_url': 'https://www.football-lab.jp/fksm/',
        'official_x_account': {'url': 'https://x.com/fufc_staff', 'handle': '@fufc_staff'},
        'official_x_hashtags': {'club': ['#福島ユナイテッド', '#福島ユナイテッドFC']},
        'aliases': {'short_name': ['福島'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '栃木ＳＣ': {
        'league': 'J3',
        'official_site_url': 'http://www.tochigisc.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/tochigi/',
        'football_lab_url': 'https://www.football-lab.jp/to-s/',
        'official_x_account': {'url': 'https://x.com/tochigisc', 'handle': '@tochigisc'},
        'official_x_hashtags': {'club': ['#栃木ＳＣ', '#栃木SC', '#全員戦力']},
        'aliases': {'short_name': ['栃木SC'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ザスパ群馬': {
        'league': 'J3',
        'official_site_url': 'http://www.thespa.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/gunma/',
        'football_lab_url': 'https://www.football-lab.jp/gnm/',
        'official_x_account': {'url': 'https://x.com/OfficialThespa', 'handle': '@OfficialThespa'},
        'official_x_hashtags': {'club': ['#ザスパ群馬', '#thespa']},
        'aliases': {'short_name': ['群馬'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ＳＣ相模原': {
        'league': 'J3',
        'official_site_url': 'http://www.scsagamihara.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/sagamihara/',
        'football_lab_url': 'https://www.football-lab.jp/sagm/',
        'official_x_account': {'url': 'https://x.com/sc_sagamihara', 'handle': '@sc_sagamihara'},
        'official_x_hashtags': {'club': ['#SC相模原', '#SCS']},
        'aliases': {'short_name': ['相模原'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '松本山雅ＦＣ': {
        'league': 'J3',
        'official_site_url': 'https://www.yamaga-fc.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/matsumoto/',
        'football_lab_url': 'https://www.football-lab.jp/mats/',
        'official_x_account': {'url': 'https://x.com/yamagafc', 'handle': '@yamagafc'},
        'official_x_hashtags': {'club': ['#松本山雅FC', '#yamaga']},
        'aliases': {'short_name': ['松本'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ＡＣ長野パルセイロ': {
        'league': 'J3',
        'official_site_url': 'https://parceiro.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/nagano/',
        'football_lab_url': 'https://www.football-lab.jp/naga/',
        'official_x_account': {'url': 'https://x.com/NAGANO_PARCEIRO', 'handle': '@NAGANO_PARCEIRO'},
        'official_x_hashtags': {'club': ['#acnp', '#パルセイロ']},
        'aliases': {'short_name': ['長野'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ツエーゲン金沢': {
        'league': 'J3',
        'official_site_url': 'http://www.zweigen-kanazawa.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kanazawa/',
        'football_lab_url': 'https://www.football-lab.jp/kana/',
        'official_x_account': {'url': 'https://x.com/zweigen_staff', 'handle': '@zweigen_staff'},
        'official_x_hashtags': {'club': ['#ツエーゲン金沢', '#zweigen']},
        'aliases': {'short_name': ['金沢'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ＦＣ岐阜': {
        'league': 'J3',
        'official_site_url': 'http://www.fc-gifu.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/gifu/',
        'football_lab_url': 'https://www.football-lab.jp/gifu/',
        'official_x_account': {'url': 'https://x.com/fcgifuDREAM', 'handle': '@fcgifuDREAM'},
        'official_x_hashtags': {'club': ['#FC岐阜', '#fcgifu']},
        'aliases': {'short_name': ['岐阜'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'レイラック滋賀ＦＣ': {
        'league': 'J3',
        'official_site_url': 'https://reilac-shiga.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/shiga/',
        'football_lab_url': 'https://www.football-lab.jp/rsfc/',
        'official_x_account': {'url': 'https://x.com/reilacshiga', 'handle': '@reilacshiga'},
        'official_x_hashtags': {'club': ['#レイラック滋賀']},
        'aliases': {'short_name': ['滋賀'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ＦＣ大阪': {
        'league': 'J3',
        'official_site_url': 'https://fc-osaka.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/fosaka/',
        'football_lab_url': 'https://www.football-lab.jp/f-os/',
        'official_x_account': {'url': 'https://x.com/FCosakaOfficial', 'handle': '@FCosakaOfficial'},
        'official_x_hashtags': {'club': ['#FC大阪', '#fcosaka']},
        'aliases': {'short_name': ['FC大阪'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '奈良クラブ': {
        'league': 'J3',
        'official_site_url': 'https://naraclub.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/nara/',
        'football_lab_url': 'https://www.football-lab.jp/nara/',
        'official_x_account': {'url': 'https://x.com/naraclub_info', 'handle': '@naraclub_info'},
        'official_x_hashtags': {'club': ['#奈良クラブ', '#naraclub']},
        'aliases': {'short_name': ['奈良'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ガイナーレ鳥取': {
        'league': 'J3',
        'official_site_url': 'https://www.gainare.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/tottori/',
        'football_lab_url': 'https://www.football-lab.jp/totr/',
        'official_x_account': {'url': 'https://x.com/gainareofficial', 'handle': '@gainareofficial'},
        'official_x_hashtags': {'club': ['#ガイナーレ鳥取', '#gainare']},
        'aliases': {'short_name': ['鳥取'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'レノファ山口ＦＣ': {
        'league': 'J3',
        'official_site_url': 'http://www.renofa.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/yamaguchi/',
        'football_lab_url': 'https://www.football-lab.jp/r-ya/',
        'official_x_account': {'url': 'https://x.com/renofayamaguchi', 'handle': '@renofayamaguchi'},
        'official_x_hashtags': {'club': ['#レノファ山口FC', '#renofa']},
        'aliases': {'short_name': ['山口'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'カマタマーレ讃岐': {
        'league': 'J3',
        'official_site_url': 'https://www.kamatamare.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/sanuki/',
        'football_lab_url': 'https://www.football-lab.jp/sanu/',
        'official_x_account': {'url': 'https://x.com/kamatama_kouhou', 'handle': '@kamatama_kouhou'},
        'official_x_hashtags': {'club': ['#カマタマーレ讃岐', '#kamatamare']},
        'aliases': {'short_name': ['讃岐'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '愛媛ＦＣ': {
        'league': 'J3',
        'official_site_url': 'http://www.ehimefc.com/p/index.html',
        'jleague_profile_url': 'https://www.jleague.jp/club/ehime/',
        'football_lab_url': 'https://www.football-lab.jp/ehim/',
        'official_x_account': {'url': 'https://x.com/ehime_fc', 'handle': '@ehime_fc'},
        'official_x_hashtags': {'club': ['#ehimefc', '#愛媛FC']},
        'aliases': {'short_name': ['愛媛'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '高知ユナイテッドＳＣ': {
        'league': 'J3',
        'official_site_url': 'http://kochi-usc.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kochi/',
        'football_lab_url': 'https://www.football-lab.jp/kusc/',
        'official_x_account': {'url': 'https://x.com/kochi_United', 'handle': '@kochi_United'},
        'official_x_hashtags': {'club': ['#高知ユナイテッドSC']},
        'aliases': {'short_name': ['高知'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ギラヴァンツ北九州': {
        'league': 'J3',
        'official_site_url': 'https://www.giravanz.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kitakyushu/',
        'football_lab_url': 'https://www.football-lab.jp/kiky/',
        'official_x_account': {'url': 'https://x.com/Giravanz_staff', 'handle': '@Giravanz_staff'},
        'official_x_hashtags': {'club': ['#ギラヴァンツ北九州', '#giravanz']},
        'aliases': {'short_name': ['北九州'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ロアッソ熊本': {
        'league': 'J3',
        'official_site_url': 'http://roasso-k.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kumamoto/',
        'football_lab_url': 'https://www.football-lab.jp/kuma/',
        'official_x_account': {'url': 'https://x.com/roassoofficial', 'handle': '@roassoofficial'},
        'official_x_hashtags': {'club': ['#ロアッソ熊本', '#roasso']},
        'aliases': {'short_name': ['熊本'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    '鹿児島ユナイテッドＦＣ': {
        'league': 'J3',
        'official_site_url': 'http://www.kufc.co.jp/',
        'jleague_profile_url': 'https://www.jleague.jp/club/kagoshima/',
        'football_lab_url': 'https://www.football-lab.jp/kufc/',
        'official_x_account': {'url': 'https://x.com/kagoshimaufc', 'handle': '@kagoshimaufc'},
        'official_x_hashtags': {'club': ['#鹿児島ユナイテッドFC', '#kagoshimaunited']},
        'aliases': {'short_name': ['鹿児島'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },
    'ＦＣ琉球': {
        'league': 'J3',
        'official_site_url': 'http://fcryukyu.com/',
        'jleague_profile_url': 'https://www.jleague.jp/club/ryukyu/',
        'football_lab_url': 'https://www.football-lab.jp/ryuk/',
        'official_x_account': {'url': 'https://x.com/fcr_info', 'handle': '@fcr_info'},
        'official_x_hashtags': {'club': ['#FC琉球', '#fcryukyu']},
        'aliases': {'short_name': ['琉球'], 'abbreviation': [], 'nickname': [], 'custom': []},
    },}


# normalize_club_key は [3/10] にある（このセルより前の [4/10] でも使うため）。


def build_alias_index(clubs: Optional[Dict[str, ClubNode]] = None) -> Dict[str, str]:
    """正式名称と aliases から検索用の索引を作る。
    キーは正規化済みの名称、値は正式名称。同じ別名を2クラブが使っていると
    誤解決するので例外にする。"""
    clubs = J_LEAGUE_CLUBS if clubs is None else clubs
    index: Dict[str, str] = {}
    for canonical_name, node in clubs.items():
        labels = [canonical_name]
        for values in node["aliases"].values():
            labels.extend(values)
        for label in labels:
            key = normalize_club_key(label)
            if not key:
                continue
            previous = index.get(key)
            if previous is not None and previous != canonical_name:
                raise ValueError(f"別名 {label!r} が {previous!r} と {canonical_name!r} で重複しています")
            index[key] = canonical_name
    return index


ALIAS_INDEX: Dict[str, str] = build_alias_index()


def resolve_club(name_or_alias: str, clubs: Optional[Dict[str, ClubNode]] = None) -> ClubNode:
    """正式名称または登録済み別名からクラブノードを取得する。見つからなければ KeyError。"""
    clubs = J_LEAGUE_CLUBS if clubs is None else clubs
    canonical = build_alias_index(clubs).get(normalize_club_key(name_or_alias))
    if canonical is None:
        raise KeyError(f"未登録のクラブ名または別名です: {name_or_alias}")
    return clubs[canonical]


def register_alias(
    club_name: str,
    alias: str,
    *,
    category: AliasCategory = "custom",
    clubs: Optional[Dict[str, ClubNode]] = None,
) -> None:
    """クラブに略称・愛称などを追加する。club_name は正式名称でも既存別名でもよい。
    他クラブが使っている別名は拒否する。"""
    clubs = J_LEAGUE_CLUBS if clubs is None else clubs
    if not alias or not alias.strip():
        raise ValueError("別名が空です")
    if category not in {"short_name", "abbreviation", "nickname", "custom"}:
        raise ValueError(f"未対応の分類です: {category}")

    alias_index = build_alias_index(clubs)
    canonical = alias_index.get(normalize_club_key(club_name))
    if canonical is None:
        raise KeyError(f"未登録のクラブ名または別名です: {club_name}")
    owner = alias_index.get(normalize_club_key(alias))
    if owner is not None and owner != canonical:
        raise ValueError(f"別名 {alias!r} は既に {owner!r} に登録されています")

    values = clubs[canonical]["aliases"][category]
    if alias not in values:
        values.append(alias)

    ALIAS_INDEX.clear()
    ALIAS_INDEX.update(build_alias_index(clubs))


def clubs_in_league(league: League, clubs: Optional[Dict[str, ClubNode]] = None) -> Dict[str, ClubNode]:
    """指定カテゴリのクラブだけを正式名称キーの辞書で返す。"""
    clubs = J_LEAGUE_CLUBS if clubs is None else clubs
    return {name: node for name, node in clubs.items() if node["league"] == league}

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [6/10] クラブ解決と設定の組み立て
# ------------------------------------------------------------
# 前のセルのクラブ情報を、この記事で使う形に薄くかぶせる層。
# クラブ名の解決、ハッシュタグの選択、大会名の自動判定をここで行う。
# ============================================================

# グラぽで実際に使う呼び方を足す。キーは正式名称でも短縮名でもよい。
# 他クラブと衝突する別名は登録されず、警告が出る。
GRAPO_EXTRA_ALIASES: Dict[str, List[str]] = {
    "名古屋": ["グランパス"],
    "鹿島": ["アントラーズ"],
    "浦和": ["レッズ"],
    "千葉": ["ジェフ"],
    "柏": ["レイソル"],
    "東京Ｖ": ["ヴェルディ"],
    "町田": ["ゼルビア"],
    "川崎Ｆ": ["フロンターレ"],
    "横浜FM": ["マリノス"],
    "清水": ["エスパルス"],
    "京都": ["サンガ"],
    "Ｇ大阪": ["ガンバ"],
    "Ｃ大阪": ["セレッソ"],
    "神戸": ["ヴィッセル"],
    "岡山": ["ファジアーノ"],
    "広島": ["サンフレッチェ"],
    "福岡": ["アビスパ"],
    "長崎": ["ヴィファーレン"],
    "札幌": ["コンサドーレ"],
    "仙台": ["ベガルタ"],
    "新潟": ["アルビレックス"],
    "湘南": ["ベルマーレ"],
    "磐田": ["ジュビロ"],
    "鳥栖": ["サガン"],
    "大分": ["トリニータ"],
    "熊本": ["ロアッソ"],
}

# 記事タイトルのハッシュタグは #grampus #vvaren のようなラテン文字表記を使うため、
# 候補が複数あるときは ASCII のタグを優先する。
HASHTAG_PREFER_ASCII: Final[bool] = True

# 大会名を自動で組み立てるときの接頭辞。
COMPETITION_PREFIX: Final[str] = "明治安田"


def apply_extra_aliases(report: Optional[RunReport] = None) -> int:
    """GRAPO_EXTRA_ALIASES をクラブ情報へ反映する。衝突は警告にして続行する。"""
    registered = 0
    for club_name, aliases in GRAPO_EXTRA_ALIASES.items():
        for alias in aliases:
            try:
                register_alias(club_name, alias, category="nickname")
                registered += 1
            except (KeyError, ValueError) as error:
                emit_warning(report, "別名を登録できませんでした（%s → %s）: %s", club_name, alias, error)
    logger.debug("独自の別名を %s件 登録しました", registered)
    return registered


@dataclass(frozen=True)
class ClubInfo:
    """1クラブぶんの情報を、この記事で使う項目だけ平らにしたもの。"""

    canonical_name: str
    league: str
    official_site_url: str
    jleague_profile_url: str
    football_lab_url: str
    x_url: str
    x_handle: str
    hashtags: Tuple[str, ...]

    @property
    def primary_hashtag(self) -> str:
        """記事タイトルに入れるハッシュタグ（先頭の # は含まない）。"""
        candidates = [tag.lstrip("#").strip() for tag in self.hashtags if tag.strip("# ")]
        if not candidates:
            return ""
        if HASHTAG_PREFER_ASCII:
            for tag in candidates:
                if tag.isascii():
                    return tag
        return candidates[0]

    @property
    def reference_links(self) -> List[Tuple[str, str]]:
        return [
            (f"{self.canonical_name} 公式サイト", self.official_site_url),
            (f"{self.canonical_name} Jリーグ公式プロフィール", self.jleague_profile_url),
            (f"{self.canonical_name} Football LAB", self.football_lab_url),
            (f"{self.canonical_name} 公式X {self.x_handle}", self.x_url),
        ]


def lookup_club(name: str, report: Optional[RunReport] = None) -> Optional[ClubInfo]:
    """クラブ名または別名から ClubInfo を引く。見つからなければ None を返して警告する。
    resolve_club は例外を投げるので、実行を止めたくないここで受け止める。"""
    if not str(name or "").strip():
        return None
    try:
        node = resolve_club(name)
    except KeyError:
        emit_warning(
            report,
            "クラブ %r がクラブ情報に見つかりません。正式名称で入力するか、GRAPO_EXTRA_ALIASES に別名を追加してください",
            name,
        )
        return None
    canonical = build_alias_index()[normalize_club_key(name)]
    account = node["official_x_account"]
    return ClubInfo(
        canonical_name=canonical,
        league=node["league"],
        official_site_url=node["official_site_url"],
        jleague_profile_url=node["jleague_profile_url"],
        football_lab_url=node["football_lab_url"],
        x_url=account["url"],
        x_handle=account["handle"],
        hashtags=tuple(node["official_x_hashtags"]["club"]),
    )


def search_name_variants(name: str) -> List[str]:
    """[4/10] の同名関数を、クラブ情報を使う版で置き換える。
    「グランパス」から「名古屋グランパス」「名古屋」まで広げられるようになる。"""
    variants = [str(name or "").strip()]
    club = lookup_club(name)
    if club is not None:
        variants.append(club.canonical_name)
        variants.extend(J_LEAGUE_CLUBS[club.canonical_name]["aliases"]["short_name"])
    return [variant for variant in dict.fromkeys(variants) if variant]


def search_league_hint(name: str) -> Optional[str]:
    """[4/10] の同名関数を、クラブ情報から答える版で置き換える。
    自チームのカテゴリが分かるので、日程一覧を1本だけ引けばよくなる。"""
    club = lookup_club(name)
    return club.league if club is not None else None


def build_competition_label(
    manual_value: str,
    my_club: Optional[ClubInfo],
    report: Optional[RunReport] = None,
) -> str:
    """フォームに入力があればそれを使い、空欄なら自チームのカテゴリから組み立てる。"""
    manual = str(manual_value or "").strip()
    if manual:
        return manual
    if my_club is None:
        emit_warning(report, "自チームのカテゴリが分からないため、大会名を J1 と仮定します")
        return f"{COMPETITION_PREFIX}J1リーグ"
    label = f"{COMPETITION_PREFIX}{my_club.league}リーグ"
    logger.info("大会名を自動で組み立てました: %s", label)
    return label


@dataclass(frozen=True)
class PreviewConfig:
    serial_number: str
    season: str
    competition: str
    section_no: str
    my_team: str
    opponent_team: str
    opponent_hashtag: str
    my_team_is_home: bool
    my_team_formation: str
    opponent_formation: str
    kickoff_date: Optional[date]
    kickoff_time: str
    venue_name: str
    venue_address: str
    venue_map_url: str
    broadcast: str
    weather_text: str
    weather_url: str
    attack_point_count: int
    defense_point_count: int
    include_reference_section: bool
    drive_folder_name: str
    my_club: Optional[ClubInfo] = None
    opponent_club: Optional[ClubInfo] = None
    docs_style: DocsStyleConfig = field(default_factory=DocsStyleConfig)

    # ---- 表示用の組み立て ----
    @property
    def serial_label(self) -> str:
        return f"D{self.serial_number}"

    @property
    def home_team(self) -> str:
        return self.my_team if self.my_team_is_home else self.opponent_team

    @property
    def away_team(self) -> str:
        return self.opponent_team if self.my_team_is_home else self.my_team

    @property
    def home_formation(self) -> str:
        return self.my_team_formation if self.my_team_is_home else self.opponent_formation

    @property
    def away_formation(self) -> str:
        return self.opponent_formation if self.my_team_is_home else self.my_team_formation

    @property
    def my_team_hashtag(self) -> str:
        return self.my_club.primary_hashtag if self.my_club else "grampus"

    @property
    def match_label(self) -> str:
        """「2026/27明治安田J1リーグ第7節」。タイトルと見出しで使い回す。"""
        return f"{self.season}{self.competition}第{self.section_no}節"

    @property
    def date_dotted(self) -> str:
        """タイトル括弧内の「2026.9.12」。ゼロ埋めしない。"""
        if self.kickoff_date is None:
            return "----.-.-"
        return f"{self.kickoff_date.year}.{self.kickoff_date.month}.{self.kickoff_date.day}"

    @property
    def kickoff_sentence(self) -> str:
        """表に入れる「2026年9月12日土曜日 19:00試合開始」。"""
        if self.kickoff_date is None:
            return ""
        weekday = WEEKDAY_JA[self.kickoff_date.weekday()]
        date_part = f"{self.kickoff_date.year}年{self.kickoff_date.month}月{self.kickoff_date.day}日{weekday}"
        return f"{date_part} {self.kickoff_time}試合開始".strip() if self.kickoff_time else date_part

    @property
    def document_title(self) -> str:
        """Google ドキュメントのファイル名。記事タイトルと同じ並びにする。"""
        tags = [f"#{self.my_team_hashtag}"] if self.my_team_hashtag else []
        if self.opponent_hashtag:
            tags.append(f"#{self.opponent_hashtag}")
        return (
            f"{self.season} {self.competition}第{self.section_no}節マッチプレビュー "
            f"{self.my_team} vs {self.opponent_team}"
            f"（{self.date_dotted}）{' '.join(tags)} {self.serial_label}"
        )

    @property
    def article_heading(self) -> str:
        """本文冒頭の見出し（H1）。"""
        return f"【マッチプレビュー】{self.match_label} {self.my_team} vs {self.opponent_team}"

    @property
    def reference_links(self) -> List[Tuple[str, str]]:
        links: List[Tuple[str, str]] = []
        if self.opponent_club:
            links.extend(self.opponent_club.reference_links)
        if self.my_club:
            links.append((f"{self.my_club.canonical_name} 公式サイト", self.my_club.official_site_url))
        return [(label, url) for label, url in links if url]


def resolve_display_name(raw_name: str, club: Optional[ClubInfo], use_official: bool) -> str:
    """フォームの入力をそのまま使うか、Jリーグ公式表記に揃えるか。"""
    raw = str(raw_name or "").strip()
    if not use_official or club is None:
        return raw
    if club.canonical_name != raw:
        logger.info("クラブ名を公式表記に揃えました: %s → %s", raw, club.canonical_name)
    return club.canonical_name


def build_preview_config(
    *,
    serial_number: str,
    season: str,
    competition: str,
    section_no: str,
    my_team: str,
    opponent_team: str,
    use_official_club_name: bool,
    opponent_hashtag: str,
    home_or_away: str,
    my_team_formation: str,
    opponent_formation: str,
    kickoff_date: str,
    kickoff_time: str,
    venue_name: str,
    venue_address: str,
    venue_map_url: str,
    broadcast: str,
    weather_text: str,
    weather_url: str,
    attack_point_count: int,
    defense_point_count: int,
    include_reference_section: bool,
    drive_folder_name: str,
    report: Optional[RunReport] = None,
) -> PreviewConfig:
    """フォームの生の文字列を検証済みの設定に変換する。"""
    my_club = lookup_club(my_team, report)
    opponent_club = lookup_club(opponent_team, report)

    if my_club and opponent_club and my_club.league != opponent_club.league:
        emit_warning(
            report,
            "所属カテゴリが異なります（%s=%s / %s=%s）。カップ戦なら COMPETITION に大会名を入れてください",
            my_club.canonical_name, my_club.league,
            opponent_club.canonical_name, opponent_club.league,
        )

    if not str(home_or_away or "").strip():
        emit_warning(
            report,
            "ホーム／アウェイが決まりませんでした。アウェイとして扱います。"
            "[4/10] の HOME_OR_AWAY で指定してください",
        )

    manual_hashtag = str(opponent_hashtag or "").lstrip("#").strip()
    resolved_hashtag = manual_hashtag or (opponent_club.primary_hashtag if opponent_club else "")
    if not manual_hashtag and resolved_hashtag:
        logger.info("ハッシュタグをクラブ情報から選びました: #%s", resolved_hashtag)
    elif not resolved_hashtag:
        emit_warning(report, "対戦相手のハッシュタグが決まりませんでした。タイトルは自チームのタグだけになります")

    return PreviewConfig(
        serial_number=normalize_serial_number(serial_number),
        season=str(season).strip(),
        competition=build_competition_label(competition, my_club, report),
        section_no=re.sub(r"\D", "", str(section_no)) or str(section_no).strip(),
        my_team=resolve_display_name(my_team, my_club, use_official_club_name),
        opponent_team=resolve_display_name(opponent_team, opponent_club, use_official_club_name),
        opponent_hashtag=resolved_hashtag,
        my_team_is_home=str(home_or_away).strip() == "ホーム",
        my_team_formation=str(my_team_formation).strip(),
        opponent_formation=str(opponent_formation).strip(),
        kickoff_date=parse_kickoff_date(kickoff_date),
        kickoff_time=str(kickoff_time).strip(),
        venue_name=str(venue_name).strip(),
        venue_address=str(venue_address).strip(),
        venue_map_url=str(venue_map_url).strip(),
        broadcast=str(broadcast).strip(),
        weather_text=str(weather_text).strip(),
        weather_url=str(weather_url).strip(),
        attack_point_count=max(1, int(attack_point_count or 1)),
        defense_point_count=max(1, int(defense_point_count or 1)),
        include_reference_section=bool(include_reference_section),
        drive_folder_name=str(drive_folder_name).strip(),
        my_club=my_club,
        opponent_club=opponent_club,
    )

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [7/10] クラブ公式サイトの収集
# ------------------------------------------------------------
# 通信の土台は [3/10] にある。ここはクラブ公式サイト向けの部分だけ。
# どのクラブがどんなページ構成かは統一されていないので、
# 「候補を拾って人が選ぶ」までしかやらない。本文には差し込まない。
# ============================================================


# プレビュー記事の「グッズ・イベント・アクセス情報」で毎回貼るリンクの種類。
# 見出しの文言はクラブごとに違うので、候補を並べて部分一致で拾う。
CLUB_SITE_LINK_KEYWORDS: Dict[str, Tuple[str, ...]] = {
    "アクセス": ("アクセス", "交通", "行き方", "access", "stadium"),
    "試合情報": ("試合情報", "ゲーム情報", "マッチ", "game_information", "match"),
    "グッズ": ("グッズ", "ストア", "オンラインストア", "store", "shop", "goods"),
    "イベント": ("イベント", "event"),
    "チケット": ("チケット", "ticket"),
}


class ClubSiteScraper:
    """クラブ情報の official_site_url を入口に、記事で使うリンクを集める。

    どのクラブがどんなページ構成かは統一されていないので、ここでは
    「候補を拾って人が選ぶ」までしかやらない。自動で本文に差し込まない。"""

    def __init__(self, fetcher: PageFetcher, report: Optional[RunReport] = None):
        self.fetcher = fetcher
        self.report = report

    def _warn(self, message: str, *args: Any) -> None:
        emit_warning(self.report, message, *args)

    def fetch_official_top(self, club_name: str, **kwargs: Any) -> Optional[FetchResult]:
        club = lookup_club(club_name, self.report)
        if club is None or not club.official_site_url:
            self._warn("クラブ %r の公式サイトURLが分からないため取得できません", club_name)
            return None
        return self.fetcher.fetch(club.official_site_url, **kwargs)

    def collect_reference_links(
        self,
        club_name: str,
        categories: Optional[Sequence[str]] = None,
        per_category: int = 2,
        **kwargs: Any,
    ) -> List[Tuple[str, str]]:
        """公式トップから、カテゴリごとに候補リンクを拾って平らな一覧で返す。"""
        page = self.fetch_official_top(club_name, **kwargs)
        if page is None:
            return []
        wanted = list(categories) if categories else list(CLUB_SITE_LINK_KEYWORDS)
        collected: List[Tuple[str, str]] = []
        seen: set = set()
        for category in wanted:
            keywords = CLUB_SITE_LINK_KEYWORDS.get(category)
            if not keywords:
                logger.debug("未知のリンク種別です: %s", category)
                continue
            for text, url in page.links(keywords, limit=per_category):
                if url in seen:
                    continue
                seen.add(url)
                collected.append((f"{category}｜{text}", url))
        if not collected:
            self._warn(
                "%s の公式サイトから候補リンクを拾えませんでした（%s で取得）。手で貼ってください",
                club_name, page.source,
            )
        else:
            logger.info("%s の公式サイトから %s件の候補リンクを拾いました（%s）",
                        club_name, len(collected), page.source)
        return collected

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [8/10] 記事テンプレート定義
# ------------------------------------------------------------
# 章立てを「構造データ」として組み立てる。Docs への書き込み方は
# 次のセルに閉じてあるので、記事の骨格を変えたいときはここだけ直す。
#
# 使えるブロック:
#   heading1 / heading2 / heading3 … 見出し
#   text                          … 本文（bold=True で太字）
#   placeholder                   … 後から書き換える灰色の斜体テキスト
#   bullets                       … 箇条書き
#   links                         … 「ラベル: URL」の行（URL部分がリンクになる）
#   table                         … 表
#   spacer                        … 空行
# ============================================================


def _text(content: str, bold: bool = False) -> Dict[str, Any]:
    return {"type": "text", "content": content, "bold": bold}


def _placeholder(content: str) -> Dict[str, Any]:
    return {"type": "placeholder", "content": content}


def _heading(level: int, content: str) -> Dict[str, Any]:
    return {"type": f"heading{level}", "content": content}


def build_match_guide_table(config: PreviewConfig) -> Dict[str, Any]:
    """観戦ガイドの2列表。空欄の項目はプレースホルダとして灰色で入る。"""
    placeholder_cells: set = set()

    def cell(value: str, fallback: str, row_index: int) -> str:
        if value:
            return value
        placeholder_cells.add((row_index, 1))
        return fallback

    venue_lines: List[str] = []
    if config.venue_name or config.venue_address:
        venue_lines.append(
            f"{config.venue_name}（{config.venue_address}）" if config.venue_address else config.venue_name
        )
    if config.venue_map_url:
        venue_lines.append(config.venue_map_url)
    venue_text = "\n".join(venue_lines)

    weather_lines: List[str] = []
    if config.weather_text:
        weather_lines.append(config.weather_text)
    if config.weather_url:
        weather_lines.append(config.weather_url)
    weather_text = "\n".join(weather_lines)

    rows = [
        [SectionLabel.KICKOFF, cell(config.kickoff_sentence, "（キックオフ日時を入力）", 0)],
        [SectionLabel.VENUE, cell(venue_text, "（会場名・住所・Googleマップのリンク）", 1)],
        [SectionLabel.BROADCAST, cell(config.broadcast, "（DAZN / 地上波・BS などの中継）", 2)],
        [SectionLabel.WEATHER, cell(weather_text, "（天気 / 気温 ℃前後の予想です。＋tenki.jpのリンク）", 3)],
    ]
    return {
        "type": "table",
        "rows": rows,
        "bold_columns": [0],
        "placeholder_cells": placeholder_cells,
        "detect_team_cells": False,
    }


def build_lineup_table(config: PreviewConfig) -> Dict[str, Any]:
    """先発予想の2列表。左がホーム、右がアウェイ。自チームのセルは色が付く。"""
    positions = "GK: \nDF: \nMF: \nFW: "
    rows = [
        [
            f"{config.home_team}({config.home_formation})",
            f"{config.away_team}({config.away_formation})",
        ],
        [positions, positions],
    ]
    return {
        "type": "table",
        "rows": rows,
        "bold_columns": [],
        "placeholder_cells": {(1, 0), (1, 1)},
        "detect_team_cells": True,
    }


def build_document_structure(
    config: PreviewConfig,
    extra_reference_links: Optional[Sequence[Tuple[str, str]]] = None,
) -> List[Dict[str, Any]]:
    opponent = config.opponent_team or "対戦相手"
    structure: List[Dict[str, Any]] = []

    # ---- 冒頭 ----
    structure.append(_heading(1, config.article_heading))
    structure.append(_text(SectionLabel.SPEAKER_DANKOBA, bold=True))
    structure.append(_placeholder(
        f"（リード文：前節の振り返りと、{opponent}戦の位置づけ。"
        "最後は「この試合をプレビューします。」で締める）"
    ))
    structure.append(_placeholder("（告知ポストなどのURLを貼る）"))
    structure.append({"type": "spacer"})

    # ---- 観戦ガイド ----
    structure.append(_heading(2, SectionLabel.GUIDE))
    structure.append(build_match_guide_table(config))
    structure.append({"type": "spacer"})
    structure.append(_text(SectionLabel.GOODS, bold=True))
    structure.append({"type": "bullets", "items": [
        f"交通・アクセス：{opponent}公式サイトまたはスタジアム公式のアクセス情報リンク",
        f"スタグル：{opponent}公式の試合情報ページ、スタジアムのグルメ一覧リンク",
        f"グッズ：{opponent}オンラインストア、{config.my_team}の新作・アウェイ限定グッズ",
        f"イベント：{opponent}公式の試合情報ページ",
    ]})
    structure.append(_placeholder("（上の各項目について、リンクと一言コメントを本文として書く）"))
    structure.append({"type": "spacer"})

    # ---- 両チームの状況と先発予想 ----
    structure.append(_heading(2, SectionLabel.SITUATION))
    structure.append(_text(SectionLabel.SPEAKER_DANKOBA, bold=True))
    structure.append(_placeholder(
        f"（{config.my_team}と{opponent}の順位、この試合で取りたい勝ち点の話）"
    ))
    structure.append(_text(SectionLabel.SPEAKER_DANKOBA, bold=True))
    structure.append(_placeholder("（出場停止選手の有無。Jリーグ公式の該当ニュースURL）"))
    structure.append(_placeholder("（負傷者、代表招集による欠場の予想。根拠になるURL）"))
    structure.append(_placeholder("（先発予想の前置き：前節からの入れ替えをどう見たか）"))
    structure.append(build_lineup_table(config))
    structure.append({"type": "spacer"})

    # ---- 勝ち筋 ----
    structure.append(_heading(2, SectionLabel.WIN_PATH.format(my_team=config.my_team)))
    structure.append(_text(SectionLabel.SPEAKER_DANKOBA, bold=True))
    structure.append(_placeholder(f"（{opponent}の前節ハイライト動画URL）"))
    structure.append(_placeholder(
        f"（・で始まる着眼点を2〜3行。{opponent}の特徴と、{config.my_team}の狙いどころ）"
    ))
    structure.append({"type": "spacer"})

    structure.append(_heading(3, SectionLabel.ATTACK))
    structure.append(_text(SectionLabel.SPEAKER_ATTACK, bold=True))
    for index in range(1, config.attack_point_count + 1):
        structure.append(_heading(3, f"{index}. （攻撃のポイントの見出し）"))
        structure.append(_placeholder("（本文：相手の弱みと、そこを突くための具体的な手立て）"))

    structure.append(_heading(3, SectionLabel.DEFENSE))
    structure.append(_text(SectionLabel.SPEAKER_DEFENSE, bold=True))
    for index in range(1, config.defense_point_count + 1):
        structure.append(_heading(3, f"{index}. （守備のポイントの見出し）"))
        structure.append(_placeholder("（本文：警戒する相手選手と、対応で徹底したいこと）"))
    structure.append({"type": "spacer"})

    # ---- おわりに ----
    structure.append(_heading(2, SectionLabel.CLOSING))
    structure.append(_text(SectionLabel.SPEAKER_DANKOBA, bold=True))
    structure.append(_placeholder("（締めの文）"))

    # ---- 参考リンク（執筆用。公開前に削除する） ----
    reference_links = list(config.reference_links)
    seen_urls = {url for _, url in reference_links}
    for label, url in (extra_reference_links or []):
        if url and url not in seen_urls:
            seen_urls.add(url)
            reference_links.append((label, url))
    if config.include_reference_section and reference_links:
        structure.append({"type": "spacer"})
        structure.append(_heading(2, SectionLabel.REFERENCE))
        structure.append({"type": "links", "items": reference_links})

    return structure


def summarize_structure(structure: List[Dict[str, Any]]) -> str:
    """生成前に章立てをログで確認できるようにする。"""
    lines = ["", "--- 生成する章立て ---"]
    for item in structure:
        item_type = str(item.get("type"))
        if item_type == "heading1":
            lines.append(f"  H1  {item.get('content')}")
        elif item_type == "heading2":
            lines.append(f"  H2  {item.get('content')}")
        elif item_type == "heading3":
            lines.append(f"    H3  {item.get('content')}")
        elif item_type == "table":
            rows = item.get("rows") or []
            columns = len(rows[0]) if rows else 0
            lines.append(f"      表  {len(rows)}行 × {columns}列")
        elif item_type == "bullets":
            lines.append(f"      箇条書き  {len(item.get('items') or [])}項目")
        elif item_type == "links":
            lines.append(f"      リンク  {len(item.get('items') or [])}件")
    lines.append("----------------------")
    return "\n".join(lines)

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [9/10] Google Docs 出力
# ------------------------------------------------------------
# 再試行つきAPI実行・カーソルのローカル管理・表の挿入とスタイル適用は
# Konwenga Helper Ver.9.0.0 の [7/8] から転用。
# 画像は扱わないので、Drive の公開権限まわりは丸ごと落としてある。
# 代わりに、作ったドキュメントを指定フォルダへ移す処理を足した。
# ============================================================


class DocsWriter:
    def __init__(
        self,
        docs_style: Optional[DocsStyleConfig] = None,
        my_team: str = "",
        report: Optional[RunReport] = None,
    ):
        if not _IN_COLAB:
            raise RuntimeError("Google Colab 環境専用です。")
        auth.authenticate_user()
        credentials, _ = default()
        self.docs_service = build("docs", "v1", credentials=credentials)
        self.drive_service = build("drive", "v3", credentials=credentials)
        self.docs_style = docs_style or DocsStyleConfig()
        self.my_team = my_team
        self.report = report
        self._doc_cursors: Dict[str, int] = {}
        # クラブ名の判定はクラブ情報の別名索引で行う。毎回作り直すと重いので1回だけ。
        self._alias_index: Dict[str, str] = build_alias_index()

    def _warn(self, message: str, *args: Any) -> None:
        emit_warning(self.report, message, *args)

    # ---- API 実行 --------------------------------------------
    def _is_retryable_google_error(self, exc: Exception) -> bool:
        status = getattr(getattr(exc, "resp", None), "status", None)
        if status in (429, 500, 502, 503, 504):
            return True
        text = str(exc)
        return any(
            key in text
            for key in ("rateLimitExceeded", "userRateLimitExceeded", "RESOURCE_EXHAUSTED", "Quota exceeded")
        )

    def _execute_google_request(self, request: Any, *, label: str) -> Any:
        wait_sec = self.docs_style.google_api_initial_wait_sec
        last_exception: Optional[Exception] = None
        for attempt in range(1, self.docs_style.google_api_max_retries + 1):
            try:
                result = request.execute()
                time.sleep(self.docs_style.google_api_request_interval_sec)
                return result
            except Exception as exc:
                last_exception = exc
                if not self._is_retryable_google_error(exc) or attempt >= self.docs_style.google_api_max_retries:
                    raise
                logger.warning(
                    "%s がレート制限/一時エラーのため再試行します (%s/%s, %.1f秒待機)",
                    label, attempt, self.docs_style.google_api_max_retries, wait_sec,
                )
                time.sleep(wait_sec)
                wait_sec = min(wait_sec * 2, self.docs_style.google_api_max_wait_sec)
        if last_exception is not None:
            raise last_exception

    def _docs_create(self, body: Dict[str, Any]) -> Dict[str, Any]:
        return self._execute_google_request(self.docs_service.documents().create(body=body), label="Docs作成")

    def _docs_get(self, document_id: str) -> Dict[str, Any]:
        return self._execute_google_request(
            self.docs_service.documents().get(documentId=document_id), label="Docs取得"
        )

    def _docs_batch_update(self, document_id: str, requests_body: List[Dict[str, Any]]) -> Dict[str, Any]:
        return self._execute_google_request(
            self.docs_service.documents().batchUpdate(documentId=document_id, body={"requests": requests_body}),
            label="Docs更新",
        )

    # ---- Drive フォルダ ---------------------------------------
    def ensure_folder(self, folder_name: str) -> Optional[str]:
        if not folder_name:
            return None
        escaped_name = folder_name.replace("\\", "\\\\").replace("'", "\\'")
        query = (
            "mimeType='application/vnd.google-apps.folder' and trashed=false "
            f"and name='{escaped_name}' and 'root' in parents"
        )
        try:
            found = self._execute_google_request(
                self.drive_service.files().list(q=query, fields="files(id,name)", pageSize=10),
                label="Driveフォルダ検索",
            )
            files = found.get("files", [])
            if files:
                return files[0].get("id")
            created = self._execute_google_request(
                self.drive_service.files().create(
                    body={"name": folder_name, "mimeType": "application/vnd.google-apps.folder"},
                    fields="id",
                ),
                label="Driveフォルダ作成",
            )
            logger.info("Drive にフォルダを作成しました: %s", folder_name)
            return created.get("id")
        except Exception:
            logger.exception("Driveフォルダの準備に失敗しました")
            self._warn("Drive のフォルダを準備できませんでした。ドキュメントはマイドライブ直下に残ります")
            return None

    def move_to_folder(self, document_id: str, folder_id: str) -> bool:
        try:
            self._execute_google_request(
                self.drive_service.files().update(
                    fileId=document_id, addParents=folder_id, removeParents="root", fields="id",
                ),
                label="ドキュメントの移動",
            )
            return True
        except Exception:
            logger.exception("ドキュメントの移動に失敗しました")
            self._warn("ドキュメントをフォルダへ移動できませんでした（マイドライブ直下に残ります）")
            return False

    # ---- 本文 ------------------------------------------------
    @staticmethod
    def _docs_text_len(text: str) -> int:
        # Google Docs API の index は UTF-16 code unit ベース
        return len(str(text).encode("utf-16-le")) // 2

    @staticmethod
    def _named_style_for(item_type: str) -> str:
        return {
            "heading1": "HEADING_1",
            "heading2": "HEADING_2",
            "heading3": "HEADING_3",
        }.get(item_type, "NORMAL_TEXT")

    @staticmethod
    def _hex_to_rgb_color(hex_color: str) -> Dict[str, Any]:
        digits = str(hex_color or "").strip().lstrip("#")
        if len(digits) == 3:
            digits = "".join(char * 2 for char in digits)
        if len(digits) != 6:
            digits = "000000"
        return {
            "red": int(digits[0:2], 16) / 255.0,
            "green": int(digits[2:4], 16) / 255.0,
            "blue": int(digits[4:6], 16) / 255.0,
        }

    def _append_paragraphs(
        self,
        document_id: str,
        lines: Sequence[str],
        named_style: str = "NORMAL_TEXT",
        *,
        bold: bool = False,
        italic: bool = False,
        color_hex: str = "",
        bullets: bool = False,
    ) -> None:
        """複数行をまとめて1回の batchUpdate で書き込む。"""
        body = "".join(f"{line}\n" for line in lines if line is not None)
        if not body.strip("\n"):
            body = "\n" * max(1, len(lines))
        index = self._doc_cursors.get(document_id, 1)
        length = self._docs_text_len(body)
        end_index = index + length

        requests_body: List[Dict[str, Any]] = [
            {"insertText": {"location": {"index": index}, "text": body}},
            {"updateParagraphStyle": {
                "range": {"startIndex": index, "endIndex": end_index},
                "paragraphStyle": {"namedStyleType": named_style},
                "fields": "namedStyleType",
            }},
        ]

        text_style: Dict[str, Any] = {}
        fields: List[str] = []
        if bold:
            text_style["bold"] = True
            fields.append("bold")
        if italic:
            text_style["italic"] = True
            fields.append("italic")
        if color_hex:
            text_style["foregroundColor"] = {"color": {"rgbColor": self._hex_to_rgb_color(color_hex)}}
            fields.append("foregroundColor")
        if text_style and end_index - 1 > index:
            requests_body.append({"updateTextStyle": {
                "range": {"startIndex": index, "endIndex": end_index - 1},
                "textStyle": text_style,
                "fields": ",".join(fields),
            }})

        if bullets:
            # createParagraphBullets は文字を挿入しないので index はずれない
            requests_body.append({"createParagraphBullets": {
                "range": {"startIndex": index, "endIndex": end_index},
                "bulletPreset": "BULLET_DISC_CIRCLE_SQUARE",
            }})

        self._docs_batch_update(document_id, requests_body)
        self._doc_cursors[document_id] = end_index

    def _append_link_lines(self, document_id: str, items: Sequence[Tuple[str, str]]) -> None:
        """「ラベル: URL」の行を書き、URL の部分だけハイパーリンクにする。
        Docs API は挿入しただけではリンクにならないので、範囲を計算して付け直す。"""
        pairs = [(str(label), str(url)) for label, url in items if str(url).strip()]
        if not pairs:
            return
        index = self._doc_cursors.get(document_id, 1)
        requests_body: List[Dict[str, Any]] = []
        body_parts: List[str] = []
        link_ranges: List[Tuple[int, int]] = []

        offset = index
        for label, url in pairs:
            prefix = f"{label}: "
            line = f"{prefix}{url}\n"
            url_start = offset + self._docs_text_len(prefix)
            url_end = url_start + self._docs_text_len(url)
            link_ranges.append((url_start, url_end))
            body_parts.append(line)
            offset += self._docs_text_len(line)

        body = "".join(body_parts)
        end_index = index + self._docs_text_len(body)
        requests_body.append({"insertText": {"location": {"index": index}, "text": body}})
        requests_body.append({"updateParagraphStyle": {
            "range": {"startIndex": index, "endIndex": end_index},
            "paragraphStyle": {"namedStyleType": "NORMAL_TEXT"},
            "fields": "namedStyleType",
        }})
        for (start, end), (_, url) in zip(link_ranges, pairs):
            requests_body.append({"updateTextStyle": {
                "range": {"startIndex": start, "endIndex": end},
                "textStyle": {"link": {"url": url}},
                "fields": "link",
            }})
        self._docs_batch_update(document_id, requests_body)
        self._doc_cursors[document_id] = end_index

    # ---- 表 --------------------------------------------------
    def _estimate_text_width_pt(self, text: Any) -> float:
        """セル内改行が起きない最小幅の目安。全角は約1em、半角は約0.55emで見積もる。
        複数行のセルは、いちばん長い行で測る。"""
        font_pt = self.docs_style.table_font_pt
        widest = 0.0
        for line in str(text).split("\n"):
            width = 0.0
            for char in line:
                width += font_pt if unicodedata.east_asian_width(char) in ("W", "F", "A") else font_pt * 0.55
            widest = max(widest, width)
        return widest

    def _estimate_column_widths(self, rows: Sequence[Sequence[str]]) -> List[float]:
        style = self.docs_style
        column_count = max((len(row) for row in rows), default=0)
        widths: List[float] = []
        for column_index in range(column_count):
            longest = max(
                (self._estimate_text_width_pt(row[column_index]) for row in rows if column_index < len(row)),
                default=0.0,
            )
            widths.append(max(style.table_min_column_width_pt, longest + style.table_cell_padding_pt))
        total = sum(widths)
        # ページ幅を超える場合だけ縮める。ここでは改行が発生しうる。
        if total > style.table_max_total_width_pt and total > 0:
            scale = style.table_max_total_width_pt / total
            widths = [max(style.table_min_column_width_pt, width * scale) for width in widths]
        return widths

    def _canonical_club(self, value: Any) -> Optional[str]:
        """「Ｖ・ファーレン長崎(3-4-2-1)」のような末尾の括弧を落として正式名称を引く。"""
        text = re.sub(r"[（(].*?[）)]\s*$", "", str(value or "")).strip()
        if not text:
            return None
        return self._alias_index.get(normalize_club_key(text))

    def _team_cell_kind(self, value: Any) -> Optional[str]:
        """セルの文字列がクラブ名なら "my" / "opponent" を返す。"""
        canonical = self._canonical_club(value)
        if canonical is None:
            return None
        my_canonical = self._canonical_club(self.my_team) if self.my_team else None
        return "my" if my_canonical and canonical == my_canonical else "opponent"

    def _detect_team_cells(self, rows: Sequence[Sequence[str]]) -> Dict[Tuple[int, int], str]:
        team_cells: Dict[Tuple[int, int], str] = {}
        if not rows:
            return team_cells
        for column_index, value in enumerate(rows[0]):
            if kind := self._team_cell_kind(value):
                team_cells[(0, column_index)] = kind
        return team_cells

    def _insert_table(self, document_id: str, item: Dict[str, Any]) -> None:
        rows: List[List[str]] = [list(row) for row in (item.get("rows") or [])]
        if not rows:
            self._warn("空の表が渡されたためスキップしました")
            return
        row_count = len(rows)
        column_count = max(len(row) for row in rows)
        for row in rows:
            row.extend([""] * (column_count - len(row)))

        insert_index = self._doc_cursors.get(document_id, 1)
        self._docs_batch_update(
            document_id,
            [{"insertTable": {"rows": row_count, "columns": column_count,
                              "location": {"index": insert_index}}}],
        )

        # 表のセル index は Docs 側で決まるため、ここで1回 documents.get() する。
        document = self._docs_get(document_id)
        body = document.get("body", {}).get("content", [])
        base_cursor = max(1, int(body[-1].get("endIndex", 2)) - 1) if body else insert_index
        tables = [element["table"] for element in body if "table" in element]
        table = tables[-1] if tables else None
        if not table:
            self._warn("挿入した表をDocs上で見つけられませんでした。セルの中身が空のままになります")
            self._doc_cursors[document_id] = base_cursor
            return

        cells = [
            (cell_data["content"][0].get("startIndex"), rows[row_index][column_index])
            for row_index, row_data in enumerate(table.get("tableRows", [])) if row_index < row_count
            for column_index, cell_data in enumerate(row_data.get("tableCells", []))
            if column_index < column_count
            and rows[row_index][column_index]
            and cell_data["content"][0].get("startIndex")
        ]
        insert_requests: List[Dict[str, Any]] = [
            {"insertText": {"location": {"index": index}, "text": text}}
            for index, text in sorted(cells, key=lambda pair: pair[0], reverse=True)
        ]
        insert_requests.append({"insertText": {"endOfSegmentLocation": {}, "text": "\n"}})
        self._docs_batch_update(document_id, insert_requests)
        added_units = sum(self._docs_text_len(text) for _, text in cells) + 1
        self._doc_cursors[document_id] = base_cursor + added_units

        try:
            self._style_table(document_id, rows, item)
        except Exception:
            logger.exception("表のスタイル適用に失敗しました（内容は挿入済み）")
            self._warn("表の書式設定に失敗しました（内容は挿入済み）")

    def _style_table(self, document_id: str, rows: List[List[str]], item: Dict[str, Any]) -> None:
        # セルの index は文字挿入後にずれるため、ここで取り直す。
        document = self._docs_get(document_id)
        body = document.get("body", {}).get("content", [])
        table_elements = [element for element in body if "table" in element]
        if not table_elements:
            return
        element = table_elements[-1]
        table_start = element.get("startIndex")
        table_end = element.get("endIndex")
        table = element["table"]
        if table_start is None or table_end is None:
            return

        start_location = {"index": table_start}
        style_requests: List[Dict[str, Any]] = []

        if self.docs_style.table_align_center:
            style_requests.append({"updateParagraphStyle": {
                "range": {"startIndex": table_start, "endIndex": table_end},
                "paragraphStyle": {"alignment": "CENTER"},
                "fields": "alignment",
            }})

        for column_index, width in enumerate(self._estimate_column_widths(rows)):
            style_requests.append({"updateTableColumnProperties": {
                "tableStartLocation": start_location,
                "columnIndices": [column_index],
                "tableColumnProperties": {
                    "widthType": "FIXED_WIDTH",
                    "width": {"magnitude": round(width, 1), "unit": "PT"},
                },
                "fields": "widthType,width",
            }})

        table_rows = table.get("tableRows", [])

        def cell_range(row_index: int, column_index: int) -> Optional[Tuple[int, int]]:
            if row_index >= len(table_rows):
                return None
            table_cells = table_rows[row_index].get("tableCells", [])
            if column_index >= len(table_cells):
                return None
            content = table_cells[column_index].get("content", [])
            if not content:
                return None
            start = content[0].get("startIndex")
            end = content[-1].get("endIndex")
            if start is None or end is None or end - 1 <= start:
                return None
            return start, end - 1

        # チーム名セルの背景色
        team_cells = self._detect_team_cells(rows) if item.get("detect_team_cells") else {}
        for (row_index, column_index), kind in team_cells.items():
            background = self.docs_style.my_team_cell_bg if kind == "my" else self.docs_style.opponent_team_cell_bg
            style_requests.append({"updateTableCellStyle": {
                "tableRange": {
                    "tableCellLocation": {
                        "tableStartLocation": start_location,
                        "rowIndex": row_index,
                        "columnIndex": column_index,
                    },
                    "rowSpan": 1,
                    "columnSpan": 1,
                },
                "tableCellStyle": {"backgroundColor": {"color": {"rgbColor": self._hex_to_rgb_color(background)}}},
                "fields": "backgroundColor",
            }})
            if kind != "my":
                continue
            # 濃い背景色なので文字色を明るくする
            if bounds := cell_range(row_index, column_index):
                style_requests.append({"updateTextStyle": {
                    "range": {"startIndex": bounds[0], "endIndex": bounds[1]},
                    "textStyle": {
                        "foregroundColor": {
                            "color": {"rgbColor": self._hex_to_rgb_color(self.docs_style.my_team_cell_text)}
                        },
                        "bold": True,
                    },
                    "fields": "foregroundColor,bold",
                }})

        # 見出し列の太字
        for column_index in item.get("bold_columns") or []:
            for row_index in range(len(rows)):
                if bounds := cell_range(row_index, column_index):
                    style_requests.append({"updateTextStyle": {
                        "range": {"startIndex": bounds[0], "endIndex": bounds[1]},
                        "textStyle": {"bold": True},
                        "fields": "bold",
                    }})

        # 未入力セルは灰色の斜体にして、埋めるべき場所を目で分かるようにする
        for (row_index, column_index) in item.get("placeholder_cells") or ():
            if (row_index, column_index) in team_cells:
                continue
            if bounds := cell_range(row_index, column_index):
                style_requests.append({"updateTextStyle": {
                    "range": {"startIndex": bounds[0], "endIndex": bounds[1]},
                    "textStyle": {
                        "foregroundColor": {
                            "color": {"rgbColor": self._hex_to_rgb_color(self.docs_style.placeholder_color)}
                        },
                        "italic": self.docs_style.placeholder_italic,
                    },
                    "fields": "foregroundColor,italic",
                }})

        if style_requests:
            self._docs_batch_update(document_id, style_requests)

    # ---- 組み立て --------------------------------------------
    def create_document(self, title: str, structure: List[Dict[str, Any]]) -> Tuple[str, str]:
        document_id = self._docs_create({"title": title}).get("documentId")
        if not document_id:
            raise RuntimeError("Google Docs の documentId を取得できませんでした")
        self._doc_cursors[document_id] = 1
        document_url = f"https://docs.google.com/document/d/{document_id}/edit"

        for item in structure:
            item_type = str(item.get("type"))
            if item_type == "table":
                self._insert_table(document_id, item)
            elif item_type == "links":
                self._append_link_lines(document_id, list(item.get("items") or []))
            elif item_type == "bullets":
                self._append_paragraphs(document_id, list(item.get("items") or []), bullets=True)
            elif item_type == "spacer":
                self._append_paragraphs(document_id, [""])
            elif item_type == "placeholder":
                self._append_paragraphs(
                    document_id, [str(item.get("content"))],
                    italic=self.docs_style.placeholder_italic,
                    color_hex=self.docs_style.placeholder_color,
                )
            else:
                self._append_paragraphs(
                    document_id, [str(item.get("content"))],
                    self._named_style_for(item_type),
                    bold=bool(item.get("bold")),
                )

        return document_id, document_url

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Dankoba Helper Ver.1.6.1  [10/10] 実行
# ------------------------------------------------------------
# 章立てをログに出してから Google ドキュメントを作る。
# 何が未入力のまま残ったかは、実行結果サマリと本文の灰色斜体で分かる。
# ============================================================


def collect_club_site_links(
    config: PreviewConfig,
    report: RunReport,
) -> List[Tuple[str, str]]:
    """相手クラブの公式サイトから候補リンクを拾う。既定では動かさない。"""
    if not SCRAPE_CLUB_SITE:
        report.skip("公式サイトの収集", "SCRAPE_CLUB_SITE が False")
        return []
    if config.opponent_club is None:
        report.skip("公式サイトの収集", "対戦相手がクラブ情報に無い")
        return []

    network = NetworkConfig(
        respect_robots=bool(RESPECT_ROBOTS_TXT),
        min_interval_sec=float(REQUEST_MIN_INTERVAL_SEC or 0),
    )
    fetcher = PageFetcher(build_session(network), network, report)
    scraper = ClubSiteScraper(fetcher, report)
    links = scraper.collect_reference_links(
        config.opponent_club.canonical_name,
        use_playwright_fallback=bool(USE_PLAYWRIGHT_FALLBACK),
        min_html_length=2000 if USE_PLAYWRIGHT_FALLBACK else 0,
    )
    if links:
        report.ok("公式サイトの収集", f"候補リンク {len(links)}件")
    else:
        report.ng("公式サイトの収集", "候補リンクを拾えませんでした")
    return links


def retry_autofill_if_needed(report: RunReport) -> None:
    """[4/10] の自動取得が空振りしていたら、ここで取り直す。

    [4/10] はクラブ情報より前に走るので、「グランパス」のような愛称では
    日程一覧と突き合わせられない。このセルまで来ればクラブ解決が使えるので、
    search_name_variants が正式名称と短縮名まで広げてくれる。
    取れた値は [4/10] の入力欄にも反映されるので、画面を見れば分かる。"""
    if not AUTO_FILL_FROM_JLEAGUE or JLEAGUE_MATCH_INFO is not None:
        return
    logger.info("クラブ情報を読み込んだので、Jリーグ公式からの取得をやり直します")
    if populate_settings_form(SETTINGS, MY_TEAM, OPPONENT_TEAM) is None:
        report.warn(
            "Jリーグ公式から試合情報を取得できませんでした。"
            "[4/10] の MATCH_PAGE_URL に試合ページのURLを貼るか、入力欄に手で入れてください"
        )
    else:
        report.ok("Jリーグ公式から取得", "2回目の試行で成功")


def main() -> RunReport:
    report = RunReport()
    apply_extra_aliases(report)
    retry_autofill_if_needed(report)

    match_settings = SETTINGS.values()
    SETTINGS.log_checklist()
    if missing := SETTINGS.missing_fields():
        report.ng("入力チェック", f"要入力 {len(missing)}件: {'、'.join(missing)}")
    else:
        report.ok("入力チェック", "必要な項目はすべて入力済み")

    config = build_preview_config(
        serial_number=match_settings["serial_number"],
        season=SEASON,
        competition=COMPETITION,
        section_no=match_settings["section_no"],
        my_team=MY_TEAM,
        opponent_team=OPPONENT_TEAM,
        use_official_club_name=USE_OFFICIAL_CLUB_NAME,
        opponent_hashtag=OPPONENT_HASHTAG,
        home_or_away=match_settings["home_or_away"],
        my_team_formation=match_settings["my_team_formation"],
        opponent_formation=match_settings["opponent_formation"],
        kickoff_date=match_settings["kickoff_date"],
        kickoff_time=match_settings["kickoff_time"],
        venue_name=match_settings["venue_name"],
        venue_address=match_settings["venue_address"],
        venue_map_url=match_settings["venue_map_url"],
        broadcast=match_settings["broadcast"],
        weather_text=match_settings["weather_text"],
        weather_url=match_settings["weather_url"],
        attack_point_count=ATTACK_POINT_COUNT,
        defense_point_count=DEFENSE_POINT_COUNT,
        include_reference_section=INCLUDE_REFERENCE_SECTION,
        drive_folder_name=DRIVE_FOLDER_NAME,
        report=report,
    )

    logger.info("Dankoba Helper Ver.%s（クラブ情報 %s 時点）", VERSION, DATA_AS_OF)
    logger.info("タイトル: %s", config.document_title)
    if config.opponent_club:
        logger.info(
            "対戦相手: %s（%s） 公式 %s / X %s",
            config.opponent_club.canonical_name,
            config.opponent_club.league,
            config.opponent_club.official_site_url,
            config.opponent_club.x_handle,
        )

    if not config.serial_number:
        report.warn("通し番号が空です。タイトルが 'D' で終わっています")

    scraped_links = collect_club_site_links(config, report)
    structure = build_document_structure(config, extra_reference_links=scraped_links)
    logger.info(summarize_structure(structure))

    try:
        writer = DocsWriter(
            docs_style=config.docs_style,
            my_team=config.my_team,
            report=report,
        )
    except Exception:
        logger.exception("Google API の初期化に失敗しました")
        report.ng("Google認証", "Docs/Driveに接続できませんでした")
        report.emit()
        return report

    try:
        document_id, document_url = writer.create_document(config.document_title, structure)
        report.document_url = document_url
        heading_count = sum(1 for item in structure if str(item.get("type")).startswith("heading"))
        table_count = sum(1 for item in structure if item.get("type") == "table")
        report.ok("テンプレート生成", f"見出し {heading_count}個 / 表 {table_count}個")
    except Exception:
        logger.exception("Google ドキュメントの生成に失敗しました")
        report.ng("テンプレート生成", config.document_title)
        report.emit()
        return report

    reference_count = sum(len(item.get("items") or []) for item in structure if item.get("type") == "links")
    if reference_count:
        report.ok("参考リンク", f"{reference_count}件（公開前に削除）")

    if config.drive_folder_name:
        folder_id = writer.ensure_folder(config.drive_folder_name)
        if folder_id and writer.move_to_folder(document_id, folder_id):
            report.ok("フォルダへ移動", config.drive_folder_name)
        else:
            report.ng("フォルダへ移動", config.drive_folder_name)
    else:
        report.skip("フォルダへ移動", "フォルダ名が未指定")

    report.emit()
    return report


if __name__ == "__main__":
    main()